# Data

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import statsmodels.api as sm
from datetime import datetime, timedelta

import requests
import time
import glob
import os
import re

from IPython.display import display

In [2]:
offset = 0
today_str = (datetime.today() - timedelta(days=offset)).strftime("%m%d") 

dtype_map = {
    "agg_pjm_ExternalNodeID": "int64",
    "pjm_ExternalNodeID": "int64",
    "agg_NodeName": "string",
    "NodeName": "string",
    "forecast_area": "string",
    "is_valid": "int8"
}

hourly_load = pd.read_csv(
    f"data/hourly_load_MW_by_distribution/hourly_load_MW_by_distribution_{today_str}.csv",
    skiprows=[1],
    dtype=dtype_map,
    parse_dates=["datetime_ending_ept", "insert_datetime"]
)

hourly_load = hourly_load.sort_values("datetime_ending_ept").reset_index(drop=True)

# PJM update nodename
hourly_load = hourly_load[hourly_load["datetime_ending_ept"] >= "2026-03-12"].copy()

In [3]:
node_cty_map = pd.read_csv('data/pnode_county_mapping.csv')
node_cty_map.columns = (
    node_cty_map.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

In [4]:
hourly_load.columns = (
    hourly_load.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

hourly_load = hourly_load.drop(columns=[
    "autokey",
    "insert_datetime"
], errors="ignore")

hourly_load["distribution_factor"] = (
    hourly_load["distribution_factor"]
    .astype(float)
    .round(8)
)

hourly_load = hourly_load.dropna(subset=[
    "datetime_ending_ept",
    "nodename",
    "distributed_load_mw"
])

hourly_load = hourly_load.drop_duplicates(
    subset=["datetime_ending_ept", "nodename"]
)


In [5]:
hourly_load.head()

,datetime_ending_ept,agg_pjm_externalnodeid,agg_nodename,pjm_externalnodeid,nodename,distribution_factor,forecast_area,forecast_load_mw,distributed_load_mw,is_valid
0,2026-04-01,8445784,AEP,32411713,JOHNSONM138 KV T1,-0.00004,AEP,15108.0,-0.58921,1
1,2026-04-01,34964545,DOM,34886335,OAKGROV435 KV TX1,0.00006,DOMINION,14019.0,0.85516,1
2,2026-04-01,34964545,DOM,1183224859,OAKGROV435 KV TX2,0.00052,DOMINION,14019.0,7.23380,1
3,2026-04-01,34964545,DOM,34885947,OAKRIDGE13 KV TX1,0.00037,DOMINION,14019.0,5.24311,1
4,2026-04-01,34964545,DOM,1305122802,OAKRIDGE13 KV TX2,0.00023,DOMINION,14019.0,3.23839,1


In [6]:
folder = "//GC-NAS/QuantTeam/cshi/XDAI_daily_constr"

files = sorted(
    glob.glob(os.path.join(folder, "DART_constr_*_morning.csv")) +
    glob.glob(os.path.join(folder, "DART_constr_*_afternoon.csv"))
)

# make sure same-date morning comes before afternoon
def file_sort_key(f):
    base = os.path.basename(f)
    m = re.search(r"DART_constr_(\d{8})_(morning|afternoon)\.csv$", base)
    file_dt = pd.to_datetime(m.group(1), format="%Y%m%d")
    run_type = m.group(2)
    run_order = {"morning": 0, "afternoon": 1}[run_type]
    return file_dt, run_order

files = sorted(files, key=file_sort_key)

dfs = []
last_file = files[-1]

for f in files:
    base = os.path.basename(f)

    m = re.search(r"DART_constr_(\d{8})_(morning|afternoon)\.csv$", base)
    file_dt = pd.to_datetime(m.group(1), format="%Y%m%d")

    df = pd.read_csv(f)

    # identify contiguous RT chunks
    is_rt = df["MARKET_TYPE_CODE"].eq("RT")
    chunk_id = (is_rt & ~is_rt.shift(fill_value=False)).cumsum()

    df_rt = df[is_rt].copy()
    df_rt["rt_chunk"] = chunk_id[is_rt].values

    # RT chunks are numbered only among RT chunks
    rt_chunk_map = {
        old: new
        for new, old in enumerate(sorted(df_rt["rt_chunk"].unique()), start=1)
    }
    df_rt["rt_chunk"] = df_rt["rt_chunk"].map(rt_chunk_map)

    # keep first RT chunk from every file
    df_keep = df_rt[df_rt["rt_chunk"].eq(1)].copy()
    df_keep.insert(0, "file_date", (file_dt - pd.Timedelta(days=1)).date())

    dfs.append(df_keep)

    # also keep second RT chunk from the latest file as today's data
    if f == last_file:
        df_today = df_rt[df_rt["rt_chunk"].eq(2)].copy()
        df_today.insert(0, "file_date", file_dt.date())
        dfs.append(df_today)

RT_constr = pd.concat(dfs, ignore_index=True)

RT_constr = RT_constr.drop(columns="rt_chunk")

# remove duplicates, keeping the later-read file
# afternoon overwrites morning when same records exist
dedup_cols = [c for c in RT_constr.columns]

RT_constr = (
    RT_constr
    .drop_duplicates(subset=dedup_cols, keep="last")
    .reset_index(drop=True)
)

RT_constr.tail()

,file_date,MARKET_TYPE_CODE,MONITORED_FACILITY_DESC,CONTINGENCY_FACILITY_DESC,SHADOW_PRICE,INTF_Name,IsNew,1,2,3,...,16,17,18,19,20,21,22,23,24,VALUE_DATETIME
846,2026-05-28,RT,LADYSMTH H1T575 CB 500 KV,LINE 500 KV MORRISVL-SPOTSLV 594A;MORRISVL 500...,-249.199997,LADYSMTH H1T575 CB 500 KV L/O LINE...,Old,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-05-28 08:10:00
847,2026-05-28,RT,Paradise - BR Tap 161 kV l/o Gibson - Albion 3...,BASE,-182.210007,Paradise - BR Tap 161 kV l/o Gibson - Albion 3...,Old,NaN,15.18,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-05-28 01:40:00
848,2026-05-28,RT,CONASTON-PEACHBOT 5012 B 500 KV,CONASTON 500 KV CONASTON 5013/500-2 GCB L;CONA...,-62.630001,CONASTON-PEACHBOT 5012 B 500 KV L/O CONA...,Old,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-05-28 05:15:00
849,2026-05-28,RT,CONASTON-HUNTERST 5013 B 500 KV,CONASTON 500 KV CONASTON 5012/500-4 GCB B;CONA...,-39.009998,CONASTON-HUNTERST 5013 B 500 KV L/O CONA...,Old,NaN,NaN,0.10,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-05-28 03:20:00
850,2026-05-28,RT,FALLS -STEELTAP FAL-STE A 138 KV,BURLINGT 230 KV BURLINGT 1-5 CB;BURLINGT 230 ...,-19.440001,FALLS -STEELTAP FAL-STE A 138 KV L/O BURL...,Old,NaN,NaN,2.31,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-05-28 02:55:00


In [7]:
folder = "//GC-NAS/QuantTeam/cshi/XDAI_daily_constr"

files = sorted(
    glob.glob(os.path.join(folder, "DART_constr_*_morning.csv")) +
    glob.glob(os.path.join(folder, "DART_constr_*_afternoon.csv"))
)

# make sure same-date morning comes before afternoon
def file_sort_key(f):
    base = os.path.basename(f)
    m = re.search(r"DART_constr_(\d{8})_(morning|afternoon)\.csv$", base)
    file_dt = pd.to_datetime(m.group(1), format="%Y%m%d")
    run_type = m.group(2)
    run_order = {"morning": 0, "afternoon": 1}[run_type]
    return file_dt, run_order

files = sorted(files, key=file_sort_key)

dfs = []

# for f in files:
#     base = os.path.basename(f)

#     m = re.search(r"DART_constr_(\d{8})_(morning|afternoon)\.csv$", base)
#     file_dt = pd.to_datetime(m.group(1), format="%Y%m%d")

#     df = pd.read_csv(f)

#     # identify contiguous DA chunks
#     is_da = df["MARKET_TYPE_CODE"].eq("DA")
#     chunk_id = (is_da & ~is_da.shift(fill_value=False)).cumsum()

#     df_da = df[is_da].copy()
#     df_da["da_chunk"] = chunk_id[is_da].values

#     # DA chunks are numbered only among DA chunks
#     da_chunk_map = {
#         old: new
#         for new, old in enumerate(sorted(df_da["da_chunk"].unique()), start=1)
#     }
#     df_da["da_chunk"] = df_da["da_chunk"].map(da_chunk_map)

#     # keep first DA chunk from every file
#     df_keep = df_da[df_da["da_chunk"].eq(1)].copy()
    
#     run_type = m.group(2)

#     if run_type == "morning":
#         file_date_val = file_dt
#     else:  # afternoon
#         file_date_val = file_dt + pd.Timedelta(days=1)

#     df_keep.insert(0, "file_date", file_date_val.date())

#     dfs.append(df_keep)


##################### morning in the afternoon ##############################
for i, f in enumerate(files):
    base = os.path.basename(f)

    m = re.search(r"DART_constr_(\d{8})_(morning|afternoon)\.csv$", base)
    file_dt = pd.to_datetime(m.group(1), format="%Y%m%d")

    # regard the last file as morning, even if filename says afternoon
    run_type = "morning" if i == len(files) - 1 else m.group(2)

    df = pd.read_csv(f)

    # identify contiguous DA chunks
    is_da = df["MARKET_TYPE_CODE"].eq("DA")
    chunk_id = (is_da & ~is_da.shift(fill_value=False)).cumsum()

    df_da = df[is_da].copy()
    df_da["da_chunk"] = chunk_id[is_da].values

    da_chunk_map = {
        old: new
        for new, old in enumerate(sorted(df_da["da_chunk"].unique()), start=1)
    }
    df_da["da_chunk"] = df_da["da_chunk"].map(da_chunk_map)

    df_keep = df_da[df_da["da_chunk"].eq(1)].copy()

    if run_type == "morning":
        file_date_val = file_dt
    else:
        file_date_val = file_dt + pd.Timedelta(days=1)

    df_keep.insert(0, "file_date", file_date_val.date())

    dfs.append(df_keep)
#########################################################################

DA_constr = pd.concat(dfs, ignore_index=True)

DA_constr = DA_constr.drop(columns="da_chunk")

dedup_cols = [
    "file_date",
    "MARKET_TYPE_CODE",
    "MONITORED_FACILITY_DESC",
    "CONTINGENCY_FACILITY_DESC",
    "INTF_Name"
]

DA_constr = (
    DA_constr
    .drop_duplicates(subset=dedup_cols, keep="last")
    .reset_index(drop=True)
)

DA_constr.tail()

,file_date,MARKET_TYPE_CODE,MONITORED_FACILITY_DESC,CONTINGENCY_FACILITY_DESC,SHADOW_PRICE,INTF_Name,IsNew,1,2,3,...,16,17,18,19,20,21,22,23,24,VALUE_DATETIME
1660,2026-05-28,DA,ASHBURN-GOOSECRE 227D B 230 KV,L230.Beaumead-Ashburn-PleasantView.274,-409.890015,ASHBURN-GOOSECRE 227D B 230 KV L/O L230...,Old,NaN,NaN,NaN,...,323.00,347.09,388.05,409.89,NaN,NaN,NaN,NaN,NaN,NaN
1661,2026-05-28,DA,ECLIPSE_115KVECL-PIN_1_LN,L345.HandsomeLake-Wayne,-415.739990,ECLIPSE_115KVECL-PIN_1_LN L/O L345.HandsomeLak...,Old,131.68,238.09,415.74,...,NaN,NaN,NaN,20.03,57.77,NaN,29.96,NaN,NaN,NaN
1662,2026-05-28,DA,PLEASNTV-ASHBURN 274D A 230 KV,L230.GooseCreek-Ashburn-Beaumeade.227,-1301.599976,PLEASNTV-ASHBURN 274D A 230 KV L/O L230...,Old,463.31,456.84,412.06,...,1021.21,1213.41,1255.84,1058.78,1301.60,1292.37,874.30,500.84,NaN,NaN
1663,2026-05-28,DA,REMINGTN_T1_230TX3_XF,L230.Remington-RemingtonCT.2077,-92.839996,REMINGTN_T1_230TX3_XF L/O L230.Remington-Remin...,Old,14.94,7.42,5.64,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1664,2026-05-28,DA,02LYONS_138KV02L-02F1_1_LN,ACTUAL,-4.610000,02LYONS_138KV02L-02F1_1_LN L/O ACTUAL,Old,NaN,NaN,NaN,...,1.68,2.62,0.68,1.48,NaN,NaN,NaN,NaN,NaN,NaN


# Cloud

## shadow

In [8]:
# copy
df = RT_constr[RT_constr["MONITORED_FACILITY_DESC"].str.contains("CLOUD")].copy()

# hourly columns
hour_cols = [str(i) for i in range(1, 25)]

# keep only rows that have at least one hourly value
df = df.dropna(subset=hour_cols, how="all")

df["file_date"] = pd.to_datetime(df["file_date"])

# reshape to long format
cloud_shadow_price = df.melt(
    id_vars=["MONITORED_FACILITY_DESC", "file_date"],
    value_vars=hour_cols,
    var_name="hour_ending",
    value_name="shadow_price"
)

# drop null hourly values
cloud_shadow_price = cloud_shadow_price.dropna(subset=["shadow_price"]).copy()

# convert HE to int
cloud_shadow_price["hour_ending"] = cloud_shadow_price["hour_ending"].astype(int)

# build datetime_ending_ept
cloud_shadow_price["datetime_ending_ept"] = (
    cloud_shadow_price["file_date"] + pd.to_timedelta(cloud_shadow_price["hour_ending"], unit="h")
)

# rename to match your target format
cloud_shadow_price = cloud_shadow_price.rename(
    columns={"MONITORED_FACILITY_DESC": "nodename"}
)

# reorder columns
cloud_shadow_price = cloud_shadow_price[
    ["datetime_ending_ept", "nodename", "shadow_price"]
].sort_values(["nodename", "datetime_ending_ept"]
).drop_duplicates(
    subset=["nodename", "datetime_ending_ept"],
    keep="last"
).reset_index(drop=True)

cloud_shadow_price.head()

,datetime_ending_ept,nodename,shadow_price
0,2026-04-01 13:00:00,CLOUD TX1 XFORMER L 115 KV,507.47
1,2026-04-01 14:00:00,CLOUD TX1 XFORMER L 115 KV,796.65
2,2026-04-01 15:00:00,CLOUD TX1 XFORMER L 115 KV,987.85
3,2026-04-02 12:00:00,CLOUD TX1 XFORMER L 115 KV,103.06
4,2026-04-02 13:00:00,CLOUD TX1 XFORMER L 115 KV,199.62


In [9]:
# copy
df = DA_constr[DA_constr["MONITORED_FACILITY_DESC"].str.contains("CLOUD")].copy()

# hourly columns
hour_cols = [str(i) for i in range(1, 25)]

# keep only rows that have at least one hourly value
df = df.dropna(subset=hour_cols, how="all")

df["file_date"] = pd.to_datetime(df["file_date"])

# reshape to long format
cloud_shadow_price_DA = df.melt(
    id_vars=["MONITORED_FACILITY_DESC", "file_date"],
    value_vars=hour_cols,
    var_name="hour_ending",
    value_name="shadow_price"
)

# drop null hourly values
cloud_shadow_price_DA = cloud_shadow_price_DA.dropna(subset=["shadow_price"]).copy()

# convert HE to int
cloud_shadow_price_DA["hour_ending"] = cloud_shadow_price_DA["hour_ending"].astype(int)

# build datetime_ending_ept
cloud_shadow_price_DA["datetime_ending_ept"] = (
    cloud_shadow_price_DA["file_date"] + 
    pd.to_timedelta(cloud_shadow_price_DA["hour_ending"], unit="h")
)

# rename to match your target format
cloud_shadow_price_DA = cloud_shadow_price_DA.rename(
    columns={
        "MONITORED_FACILITY_DESC": "nodename",
        "shadow_price": "shadow_price_DA"
    }
)

# reorder columns
cloud_shadow_price_DA = cloud_shadow_price_DA[
    ["datetime_ending_ept", "nodename", "shadow_price_DA"]
].sort_values(
    ["nodename", "datetime_ending_ept"]
).drop_duplicates(
    subset=["nodename", "datetime_ending_ept"],
    keep="last"
).reset_index(drop=True)

cloud_shadow_price_DA.head()

,datetime_ending_ept,nodename,shadow_price_DA
0,2026-04-03 13:00:00,CLOUD_115KVCLO-KERR_1_LN,16.93
1,2026-04-03 14:00:00,CLOUD_115KVCLO-KERR_1_LN,16.92
2,2026-04-03 15:00:00,CLOUD_115KVCLO-KERR_1_LN,21.25
3,2026-04-04 11:00:00,CLOUD_115KVCLO-KERR_1_LN,79.89
4,2026-04-04 12:00:00,CLOUD_115KVCLO-KERR_1_LN,137.62


In [10]:
cloud_shadow_price_DA['nodename'].unique()

<ArrowStringArray>
['CLOUD_115KVCLO-KERR_1_LN', 'CLOUD_T2_230TX1_XF']
Length: 2, dtype: str

## county

In [11]:
# Get all nodenames that belong to agg_nodename == "DOM" in hourly_load
dom_nodes = (
    hourly_load.loc[hourly_load["agg_nodename"] == "DOM", "nodename"]
    .dropna()
    .unique()
)

# Map those DOM nodenames to county_name using node_cty_map
dom_node_counties = (
    node_cty_map.loc[node_cty_map["nodename"].isin(dom_nodes), ["nodename", "county_name"]]
    .dropna(subset=["county_name"])
    .drop_duplicates()
)

# Get all county_name touched by those DOM nodenames
dom_counties = dom_node_counties["county_name"].unique()

# From those counties, get all nodenames in node_cty_map
all_nodes_in_dom_counties = (
    node_cty_map.loc[node_cty_map["county_name"].isin(dom_counties), "nodename"]
    .dropna()
    .unique()
)

# Select all rows from hourly_load whose nodename is in those nodenames
hourly_load_dom_counties = hourly_load.loc[
    hourly_load["nodename"].isin(all_nodes_in_dom_counties)
].copy()

In [12]:
hourly_load_county = hourly_load_dom_counties[["datetime_ending_ept", "agg_nodename", "nodename", "distribution_factor", "forecast_area", "forecast_load_mw", "distributed_load_mw"]].merge(node_cty_map[["nodename", "county_name", "latitude", "longitude", "state_short_name"]], on="nodename", how="left")

county_load = (
    hourly_load_county
    .groupby(
        ["datetime_ending_ept", "county_name", "state_short_name"],
        as_index=False
    )
    .agg(
        distributed_load_mw=("distributed_load_mw", "sum"),
        latitude=("latitude", "mean"),
        longitude=("longitude", "mean")
    )
    .sort_values(["datetime_ending_ept", "county_name"])
)

In [13]:
county_geo = (
    county_load
    .groupby("county_name", as_index=False)
    .agg(
        latitude=("latitude", "mean"),
        longitude=("longitude", "mean")
    )
)

# round to 2 decimals
county_geo["latitude"] = county_geo["latitude"].round(2)
county_geo["longitude"] = county_geo["longitude"].round(2)

county_load = county_load[["datetime_ending_ept", "county_name", "state_short_name", "distributed_load_mw"]].merge(county_geo, on="county_name", how="left").copy()

In [14]:
county_load = county_load[(county_load["datetime_ending_ept"] >= "2026-04-01")].copy()

In [15]:
county_load["datetime_ending_ept"] = pd.to_datetime(county_load["datetime_ending_ept"])

county_load_shadow_price = county_load.merge(
    cloud_shadow_price[["datetime_ending_ept", "shadow_price"]],
    on=["datetime_ending_ept"],
    how="left"
)

county_load_shadow_price["shadow_price"] = county_load_shadow_price["shadow_price"].fillna(0)

In [16]:
# row-level filter
df_filtered = county_load_shadow_price[
    (county_load_shadow_price["shadow_price"].notna()) &
    (county_load_shadow_price["shadow_price"] != 0)
].copy()

# keep only counties with mean load > 20
county_load_shadow = df_filtered[
    df_filtered.groupby("county_name")["distributed_load_mw"].transform("max") > 20
].copy()

In [17]:
df = county_load_shadow.copy()

feature_cols = ["distributed_load_mw"] 

# 3. correlation between each feature and shadow_price for each nodename
corr_by_node = (
    df.groupby("county_name")
      .apply(lambda g: g[feature_cols].corrwith(g["shadow_price"]))
      .reset_index()
)

corr_by_node.sort_values(by="distributed_load_mw", ascending=False)

,county_name,distributed_load_mw
24,VA_Caroline,0.166848
17,VA_Arlington,0.129063
37,VA_Fredericksburg,0.108874
33,VA_Falls Church,0.074534
81,VA_Waynesboro,0.064323
...,...,...
19,VA_Bedford,-0.235897
6,NC_Dare,-0.237480
5,NC_Currituck,-0.241368
56,VA_New Kent,-0.248010


### plot

In [18]:
county_load_shadow_price = county_load_shadow_price.merge(
    cloud_shadow_price_DA[["datetime_ending_ept", "shadow_price_DA"]],
    on=["datetime_ending_ept"],
    how="left"
)

county_load_shadow_price["shadow_price_DA"] = county_load_shadow_price["shadow_price_DA"].fillna(0)

In [19]:
df = county_load_shadow_price.copy()

node = "VA_Mecklenburg"

sub = df[df["county_name"] == node].copy()

# ensure datetime is sorted
sub["datetime_ending_ept"] = pd.to_datetime(sub["datetime_ending_ept"])
sub = sub.sort_values("datetime_ending_ept")

fig = go.Figure()

# Left y-axis: distributed load
fig.add_trace(
    go.Scatter(
        x=sub["datetime_ending_ept"],
        y=sub["distributed_load_mw"],
        name="Distributed Load (MW)",
        line=dict(color="blue"),
        yaxis="y1"
    )
)

# Right y-axis: shadow price
fig.add_trace(
    go.Scatter(
        x=sub["datetime_ending_ept"],
        y=sub["shadow_price"],
        name="Shadow Price",
        line=dict(color="red"),
        yaxis="y2"
    )
)

fig.add_trace(
    go.Scatter(
        x=sub["datetime_ending_ept"],
        y=sub["shadow_price_DA"],
        name="Shadow Price DA",
        line=dict(color="green"),
        yaxis="y2"
    )
)

fig.add_hline(
    y=350,
    line_dash="dash",
    line_color="black",
    annotation_text="350 MW",
    annotation_position="bottom right"
)

# Layout with dual axis
fig.update_layout(
    title=f"{node}: Load vs Shadow Price",
    xaxis=dict(title="Datetime"),
    
    yaxis=dict(
        title="Distributed Load (MW)",
        side="left"
    ),
    
    yaxis2=dict(
        title="Shadow Price",
        overlaying="y",
        side="right"
    ),
    
    legend=dict(x=0.01, y=0.99),
    height=500,
    width=2000
)

fig.show()

# Chicago

## shadow

In [20]:
# copy
df = RT_constr[RT_constr["MONITORED_FACILITY_DESC"].str.contains("Chicago")].copy()

# hourly columns
hour_cols = [str(i) for i in range(1, 25)]

# keep only rows that have at least one hourly value
df = df.dropna(subset=hour_cols, how="all")

df["file_date"] = pd.to_datetime(df["file_date"])

# reshape to long format
chicago_shadow_price = df.melt(
    id_vars=["MONITORED_FACILITY_DESC", "file_date"],
    value_vars=hour_cols,
    var_name="hour_ending",
    value_name="shadow_price"
)

# drop null hourly values
chicago_shadow_price = chicago_shadow_price.dropna(subset=["shadow_price"]).copy()

# convert HE to int
chicago_shadow_price["hour_ending"] = chicago_shadow_price["hour_ending"].astype(int)

# build datetime_ending_ept
chicago_shadow_price["datetime_ending_ept"] = (
    chicago_shadow_price["file_date"] + pd.to_timedelta(chicago_shadow_price["hour_ending"], unit="h")
)

# rename to match your target format
chicago_shadow_price = chicago_shadow_price.rename(
    columns={"MONITORED_FACILITY_DESC": "nodename"}
)

# reorder columns
chicago_shadow_price = chicago_shadow_price[
    ["datetime_ending_ept", "nodename", "shadow_price"]
].sort_values(["nodename", "datetime_ending_ept"]
).drop_duplicates(
    subset=["nodename", "datetime_ending_ept"],
    keep="last"
).reset_index(drop=True)

chicago_shadow_price.head()

,datetime_ending_ept,nodename,shadow_price
0,2026-04-01 04:00:00,Chicago - Praxair3 138 kV l/o Wilton Center - ...,16.99
1,2026-04-01 05:00:00,Chicago - Praxair3 138 kV l/o Wilton Center - ...,159.08
2,2026-04-01 06:00:00,Chicago - Praxair3 138 kV l/o Wilton Center - ...,24.06
3,2026-04-01 18:00:00,Chicago - Praxair3 138 kV l/o Wilton Center - ...,281.11
4,2026-04-01 19:00:00,Chicago - Praxair3 138 kV l/o Wilton Center - ...,159.77


In [21]:
# copy
df = DA_constr[DA_constr["MONITORED_FACILITY_DESC"].str.contains("Chicago")].copy()

# hourly columns
hour_cols = [str(i) for i in range(1, 25)]

# keep only rows that have at least one hourly value
df = df.dropna(subset=hour_cols, how="all")

df["file_date"] = pd.to_datetime(df["file_date"])

# reshape to long format
chicago_shadow_price_DA = df.melt(
    id_vars=["MONITORED_FACILITY_DESC", "file_date"],
    value_vars=hour_cols,
    var_name="hour_ending",
    value_name="shadow_price"
)

# drop null hourly values
chicago_shadow_price_DA = chicago_shadow_price_DA.dropna(subset=["shadow_price"]).copy()

# convert HE to int
chicago_shadow_price_DA["hour_ending"] = chicago_shadow_price_DA["hour_ending"].astype(int)

# build datetime_ending_ept
chicago_shadow_price_DA["datetime_ending_ept"] = (
    chicago_shadow_price_DA["file_date"] + 
    pd.to_timedelta(chicago_shadow_price_DA["hour_ending"], unit="h")
)

# rename to match your target format
chicago_shadow_price_DA = chicago_shadow_price_DA.rename(
    columns={
        "MONITORED_FACILITY_DESC": "nodename",
        "shadow_price": "shadow_price_DA"
    }
)

# reorder columns
chicago_shadow_price_DA = chicago_shadow_price_DA[
    ["datetime_ending_ept", "nodename", "shadow_price_DA"]
].sort_values(
    ["nodename", "datetime_ending_ept"]
).drop_duplicates(
    subset=["nodename", "datetime_ending_ept"],
    keep="last"
).reset_index(drop=True)

chicago_shadow_price_DA.head()

,datetime_ending_ept,nodename,shadow_price_DA
0,2026-04-03 02:00:00,Chicago-Praxair3 138 kV l/o Wilton Center-Dumo...,47.33
1,2026-04-04 02:00:00,Chicago-Praxair3 138 kV l/o Wilton Center-Dumo...,128.46
2,2026-04-04 04:00:00,Chicago-Praxair3 138 kV l/o Wilton Center-Dumo...,2.27
3,2026-04-04 06:00:00,Chicago-Praxair3 138 kV l/o Wilton Center-Dumo...,0.63
4,2026-04-04 08:00:00,Chicago-Praxair3 138 kV l/o Wilton Center-Dumo...,111.83


In [22]:
chicago_shadow_price["shadow_price"].max()

np.float64(2000.0)

## county

In [23]:
# Get all nodenames that belong to agg_nodename == "COMED" in hourly_load
ce_nodes = (
    hourly_load.loc[hourly_load["agg_nodename"] == "COMED", "nodename"]
    .dropna()
    .unique()
)

# Map those COMED nodenames to county_name using node_cty_map
ce_node_counties = (
    node_cty_map.loc[node_cty_map["nodename"].isin(ce_nodes), ["nodename", "county_name"]]
    .dropna(subset=["county_name"])
    .drop_duplicates()
)

# Get all county_name touched by those COMED nodenames
ce_counties = ce_node_counties["county_name"].unique()

# From those counties, get all nodenames in node_cty_map
all_nodes_in_ce_counties = (
    node_cty_map.loc[node_cty_map["county_name"].isin(ce_counties), "nodename"]
    .dropna()
    .unique()
)

# Select all rows from hourly_load whose nodename is in those nodenames
hourly_load_ce_counties = hourly_load.loc[
    hourly_load["nodename"].isin(all_nodes_in_ce_counties)
].copy()

In [24]:
hourly_load_county = hourly_load_ce_counties[["datetime_ending_ept", "agg_nodename", "nodename", "distribution_factor", "forecast_area", "forecast_load_mw", "distributed_load_mw"]].merge(node_cty_map[["nodename", "county_name", "latitude", "longitude", "state_short_name"]], on="nodename", how="left")

county_load = (
    hourly_load_county
    .groupby(
        ["datetime_ending_ept", "county_name", "state_short_name"],
        as_index=False
    )
    .agg(
        distributed_load_mw=("distributed_load_mw", "sum"),
        latitude=("latitude", "mean"),
        longitude=("longitude", "mean")
    )
    .sort_values(["datetime_ending_ept", "county_name"])
)

county_load = county_load[county_load["state_short_name"] == "IL"].copy()

In [25]:
county_geo = (
    county_load
    .groupby("county_name", as_index=False)
    .agg(
        latitude=("latitude", "mean"),
        longitude=("longitude", "mean")
    )
)

# round to 2 decimals
county_geo["latitude"] = county_geo["latitude"].round(2)
county_geo["longitude"] = county_geo["longitude"].round(2)

county_load = county_load[["datetime_ending_ept", "county_name", "state_short_name", "distributed_load_mw"]].merge(county_geo, on="county_name", how="left").copy()

In [26]:
county_load = county_load[(county_load["datetime_ending_ept"] >= "2026-04-01")].copy()

In [27]:
county_load["datetime_ending_ept"] = pd.to_datetime(county_load["datetime_ending_ept"])

county_load_shadow_price = county_load.merge(
    chicago_shadow_price[["datetime_ending_ept", "shadow_price"]],
    on=["datetime_ending_ept"],
    how="left"
)

county_load_shadow_price["shadow_price"] = county_load_shadow_price["shadow_price"].fillna(0)

In [28]:
# row-level filter
df_filtered = county_load_shadow_price[
    (county_load_shadow_price["shadow_price"].notna()) &
    (county_load_shadow_price["shadow_price"] != 0)
].copy()

# keep only counties with max load > 20
county_load_shadow = df_filtered[
    df_filtered.groupby("county_name")["distributed_load_mw"].transform("max") > 20
].copy()

### plot

In [29]:
county_load_shadow_price = county_load_shadow_price.merge(
    chicago_shadow_price_DA[["datetime_ending_ept", "shadow_price_DA"]],
    on=["datetime_ending_ept"],
    how="left"
)

county_load_shadow_price["shadow_price_DA"] = county_load_shadow_price["shadow_price_DA"].fillna(0)



In [30]:
df = county_load_shadow_price.copy()

node = "IL_Cook"

sub = df[df["county_name"] == node].copy()

# ensure datetime is sorted
sub["datetime_ending_ept"] = pd.to_datetime(sub["datetime_ending_ept"])
sub = sub.sort_values("datetime_ending_ept")

fig = go.Figure()

# Left y-axis: distributed load
fig.add_trace(
    go.Scatter(
        x=sub["datetime_ending_ept"],
        y=sub["distributed_load_mw"],
        name="Distributed Load (MW)",
        line=dict(color="blue"),
        yaxis="y1"
    )
)

# Right y-axis: shadow price
fig.add_trace(
    go.Scatter(
        x=sub["datetime_ending_ept"],
        y=sub["shadow_price"],
        name="Shadow Price",
        line=dict(color="red"),
        yaxis="y2"
    )
)

fig.add_trace(
    go.Scatter(
        x=sub["datetime_ending_ept"],
        y=sub["shadow_price_DA"],
        name="Shadow Price DA",
        line=dict(color="green"),
        yaxis="y2"
    )
)

# Layout with dual axis
fig.update_layout(
    title=f"{node}: Load vs Shadow Price",
    xaxis=dict(title="Datetime"),
    
    yaxis=dict(
        title="Distributed Load (MW)",
        side="left"
    ),
    
    yaxis2=dict(
        title="Shadow Price",
        overlaying="y",
        side="right"
    ),
    
    legend=dict(x=0.01, y=0.99),
    height=500,
    width=2000
)

fig.show()

# COOK

## shadow

In [8]:
# copy
df = RT_constr[
    RT_constr["MONITORED_FACILITY_DESC"].str.contains("COOK")
].copy()

# hourly columns
hour_cols = [str(i) for i in range(1, 25)]

# keep only rows that have at least one hourly value
df = df.dropna(subset=hour_cols, how="all")

df["file_date"] = pd.to_datetime(df["file_date"])

# reshape to long format
cook_shadow_price = df.melt(
    id_vars=["MONITORED_FACILITY_DESC", "file_date"],
    value_vars=hour_cols,
    var_name="hour_ending",
    value_name="shadow_price"
)

# drop null hourly values
cook_shadow_price = cook_shadow_price.dropna(
    subset=["shadow_price"]
).copy()

# convert HE to int
cook_shadow_price["hour_ending"] = (
    cook_shadow_price["hour_ending"].astype(int)
)

# build datetime_ending_ept
cook_shadow_price["datetime_ending_ept"] = (
    cook_shadow_price["file_date"]
    + pd.to_timedelta(
        cook_shadow_price["hour_ending"],
        unit="h"
    )
)

# rename to match your target format
cook_shadow_price = cook_shadow_price.rename(
    columns={"MONITORED_FACILITY_DESC": "nodename"}
)

# reorder columns
cook_shadow_price = cook_shadow_price[
    ["datetime_ending_ept", "nodename", "shadow_price"]
].sort_values(
    ["nodename", "datetime_ending_ept"]
).drop_duplicates(
    subset=["nodename", "datetime_ending_ept"],
    keep="last"
).reset_index(drop=True)

cook_shadow_price.head()

,datetime_ending_ept,nodename,shadow_price
0,2026-05-26 15:00:00,COOK-OLIVE A 345 KV,187.34
1,2026-05-26 16:00:00,COOK-OLIVE A 345 KV,239.56
2,2026-05-26 17:00:00,COOK-OLIVE A 345 KV,333.33
3,2026-05-26 18:00:00,COOK-OLIVE A 345 KV,329.82
4,2026-05-26 19:00:00,COOK-OLIVE A 345 KV,585.83


In [9]:
cook_shadow_price['shadow_price'].max()

np.float64(930.51)

In [10]:
# copy
df = DA_constr[
    DA_constr["MONITORED_FACILITY_DESC"].str.contains("COOK")
].copy()

# hourly columns
hour_cols = [str(i) for i in range(1, 25)]

# keep only rows that have at least one hourly value
df = df.dropna(subset=hour_cols, how="all")

df["file_date"] = pd.to_datetime(df["file_date"])

# reshape to long format
cook_shadow_price_DA = df.melt(
    id_vars=["MONITORED_FACILITY_DESC", "file_date"],
    value_vars=hour_cols,
    var_name="hour_ending",
    value_name="shadow_price"
)

# drop null hourly values
cook_shadow_price_DA = cook_shadow_price_DA.dropna(
    subset=["shadow_price"]
).copy()

# convert HE to int
cook_shadow_price_DA["hour_ending"] = (
    cook_shadow_price_DA["hour_ending"].astype(int)
)

# build datetime_ending_ept
cook_shadow_price_DA["datetime_ending_ept"] = (
    cook_shadow_price_DA["file_date"]
    + pd.to_timedelta(
        cook_shadow_price_DA["hour_ending"],
        unit="h"
    )
)

# rename to match your target format
cook_shadow_price_DA = cook_shadow_price_DA.rename(
    columns={
        "MONITORED_FACILITY_DESC": "nodename",
        "shadow_price": "shadow_price_DA"
    }
)

# reorder columns
cook_shadow_price_DA = cook_shadow_price_DA[
    ["datetime_ending_ept", "nodename", "shadow_price_DA"]
].sort_values(
    ["nodename", "datetime_ending_ept"]
).drop_duplicates(
    subset=["nodename", "datetime_ending_ept"],
    keep="last"
).reset_index(drop=True)

cook_shadow_price_DA.head()

,datetime_ending_ept,nodename,shadow_price_DA


## county

In [11]:
# Get all nodenames that belong to agg_nodename == "AEP" in hourly_load
aep_nodes = (
    hourly_load.loc[hourly_load["agg_nodename"].isin(["AEP"]), "nodename"]
    .dropna()
    .unique()
)

# Map those AEP nodenames to county_name using node_cty_map
aep_node_counties = (
    node_cty_map.loc[node_cty_map["nodename"].isin(aep_nodes), ["nodename", "county_name"]]
    .dropna(subset=["county_name"])
    .drop_duplicates()
)

# Get all county_name touched by those AEP nodenames
aep_counties = aep_node_counties["county_name"].unique()

# From those counties, get all nodenames in node_cty_map
all_nodes_in_aep_counties = (
    node_cty_map.loc[node_cty_map["county_name"].isin(aep_counties), "nodename"]
    .dropna()
    .unique()
)

# Select all rows from hourly_load whose nodename is in those nodenames
hourly_load_aep_counties = hourly_load.loc[
    hourly_load["nodename"].isin(all_nodes_in_aep_counties)
].copy()

In [12]:
hourly_load_county = hourly_load_aep_counties[["datetime_ending_ept", "agg_nodename", "nodename", "distribution_factor", "forecast_area", "forecast_load_mw", "distributed_load_mw"]].merge(node_cty_map[["nodename", "county_name", "latitude", "longitude", "state_short_name"]], on="nodename", how="left")

county_load = (
    hourly_load_county
    .groupby(
        ["datetime_ending_ept", "county_name", "state_short_name"],
        as_index=False
    )
    .agg(
        distributed_load_mw=("distributed_load_mw", "sum"),
        latitude=("latitude", "mean"),
        longitude=("longitude", "mean")
    )
    .sort_values(["datetime_ending_ept", "county_name"])
)

In [13]:
county_geo = (
    county_load
    .groupby("county_name", as_index=False)
    .agg(
        latitude=("latitude", "mean"),
        longitude=("longitude", "mean")
    )
)

# round to 2 decimals
county_geo["latitude"] = county_geo["latitude"].round(2)
county_geo["longitude"] = county_geo["longitude"].round(2)

county_load = county_load[["datetime_ending_ept", "county_name", "state_short_name", "distributed_load_mw"]].merge(county_geo, on="county_name", how="left").copy()

In [14]:
county_load = county_load[county_load["state_short_name"].isin(["IN", "MI"])].copy()

county_load = county_load[(county_load["datetime_ending_ept"] >= "2026-05-01")].copy()

In [15]:
county_load["datetime_ending_ept"] = pd.to_datetime(county_load["datetime_ending_ept"])

county_load_shadow_price = county_load.merge(
    cook_shadow_price[["datetime_ending_ept", "shadow_price"]],
    on=["datetime_ending_ept"],
    how="left"
)

county_load_shadow_price["shadow_price"] = county_load_shadow_price["shadow_price"].fillna(0)

In [16]:
# row-level filter
df_filtered = county_load_shadow_price[
    (county_load_shadow_price["shadow_price"].notna()) &
    (county_load_shadow_price["shadow_price"] != 0)
].copy()

# keep only counties with mean load > 20
county_load_shadow = df_filtered[
    df_filtered.groupby("county_name")["distributed_load_mw"].transform("mean") > 20
].copy()

In [17]:
df = county_load_shadow.copy()

feature_cols = ["distributed_load_mw"] 

# 3. correlation between each feature and shadow_price for each nodename
corr_by_node = (
    df.groupby("county_name")
      .apply(lambda g: g[feature_cols].corrwith(g["shadow_price"]))
      .reset_index()
)

corr_by_node.sort_values(by="distributed_load_mw", ascending=False)

,county_name,distributed_load_mw
19,MI_Cass,0.431105
21,MI_Van Buren,0.355277
15,IN_Wayne,0.351082
12,IN_Noble,0.336529
13,IN_Randolph,0.261443
18,MI_Berrien,0.257039
20,MI_St. Joseph,0.189408
1,IN_Allen,0.176885
11,IN_Madison,0.171309
5,IN_Elkhart,0.165775


### plot

In [18]:
county_load_shadow_price = county_load_shadow_price.merge(
    cook_shadow_price_DA[["datetime_ending_ept", "shadow_price_DA"]],
    on=["datetime_ending_ept"],
    how="left"
)

county_load_shadow_price["shadow_price_DA"] = county_load_shadow_price["shadow_price_DA"].fillna(0)



In [19]:
df = county_load_shadow_price.copy()

node = "MI_Berrien"

sub = df[df["county_name"] == node].copy()

# ensure datetime is sorted
sub["datetime_ending_ept"] = pd.to_datetime(sub["datetime_ending_ept"])
sub = sub.sort_values("datetime_ending_ept")

fig = go.Figure()

# Left y-axis: distributed load
fig.add_trace(
    go.Scatter(
        x=sub["datetime_ending_ept"],
        y=sub["distributed_load_mw"],
        name="Distributed Load (MW)",
        line=dict(color="blue"),
        yaxis="y1"
    )
)

# Right y-axis: shadow price
fig.add_trace(
    go.Scatter(
        x=sub["datetime_ending_ept"],
        y=sub["shadow_price"],
        name="Shadow Price",
        line=dict(color="red"),
        yaxis="y2"
    )
)


fig.add_trace(
    go.Scatter(
        x=sub["datetime_ending_ept"],
        y=sub["shadow_price_DA"],
        name="Shadow Price DA",
        line=dict(color="green"),
        yaxis="y2"
    )
)

# Layout with dual axis
fig.update_layout(
    title=f"{node}: Load vs Shadow Price",
    xaxis=dict(title="Datetime"),
    
    yaxis=dict(
        title="Distributed Load (MW)",
        side="left"
    ),
    
    yaxis2=dict(
        title="Shadow Price",
        overlaying="y",
        side="right"
    ),
    
    legend=dict(x=0.01, y=0.99),
    height=500,
    width=2000
)

fig.show()

# HYATTCS

## shadow

In [146]:
# copy
df = RT_constr[RT_constr["MONITORED_FACILITY_DESC"] == "HYATTCS-MARYSVI2 MAR-HYA2   B  345 KV"].copy()

# hourly columns
hour_cols = [str(i) for i in range(1, 25)]

# keep only rows that have at least one hourly value
df = df.dropna(subset=hour_cols, how="all")

df["file_date"] = pd.to_datetime(df["file_date"])

# reshape to long format
hyattcs_shadow_price = df.melt(
    id_vars=["MONITORED_FACILITY_DESC", "file_date"],
    value_vars=hour_cols,
    var_name="hour_ending",
    value_name="shadow_price"
)

# drop null hourly values
hyattcs_shadow_price = hyattcs_shadow_price.dropna(subset=["shadow_price"]).copy()

# convert HE to int
hyattcs_shadow_price["hour_ending"] = hyattcs_shadow_price["hour_ending"].astype(int)

# build datetime_ending_ept
hyattcs_shadow_price["datetime_ending_ept"] = (
    hyattcs_shadow_price["file_date"] + pd.to_timedelta(hyattcs_shadow_price["hour_ending"], unit="h")
)

# rename to match your target format
hyattcs_shadow_price = hyattcs_shadow_price.rename(
    columns={"MONITORED_FACILITY_DESC": "nodename"}
)

# reorder columns
hyattcs_shadow_price = hyattcs_shadow_price[
    ["datetime_ending_ept", "nodename", "shadow_price"]
].sort_values(["nodename", "datetime_ending_ept"]
).drop_duplicates(
    subset=["nodename", "datetime_ending_ept"],
    keep="last"
).reset_index(drop=True)

hyattcs_shadow_price.head()

,datetime_ending_ept,nodename,shadow_price
0,2026-04-12 18:00:00,HYATTCS-MARYSVI2 MAR-HYA2 B 345 KV,579.32
1,2026-04-12 19:00:00,HYATTCS-MARYSVI2 MAR-HYA2 B 345 KV,133.35
2,2026-04-12 21:00:00,HYATTCS-MARYSVI2 MAR-HYA2 B 345 KV,4.99
3,2026-04-12 22:00:00,HYATTCS-MARYSVI2 MAR-HYA2 B 345 KV,2.79
4,2026-04-13 14:00:00,HYATTCS-MARYSVI2 MAR-HYA2 B 345 KV,194.60


In [147]:
# copy
df = DA_constr[DA_constr["MONITORED_FACILITY_DESC"].str.contains("HYATTCS")].copy()

# hourly columns
hour_cols = [str(i) for i in range(1, 25)]

# keep only rows that have at least one hourly value
df = df.dropna(subset=hour_cols, how="all")

df["file_date"] = pd.to_datetime(df["file_date"])

# reshape to long format
hyattcs_shadow_price_DA = df.melt(
    id_vars=["MONITORED_FACILITY_DESC", "file_date"],
    value_vars=hour_cols,
    var_name="hour_ending",
    value_name="shadow_price"
)

# drop null hourly values
hyattcs_shadow_price_DA = hyattcs_shadow_price_DA.dropna(subset=["shadow_price"]).copy()

# convert HE to int
hyattcs_shadow_price_DA["hour_ending"] = hyattcs_shadow_price_DA["hour_ending"].astype(int)

# build datetime_ending_ept
hyattcs_shadow_price_DA["datetime_ending_ept"] = (
    hyattcs_shadow_price_DA["file_date"] + 
    pd.to_timedelta(hyattcs_shadow_price_DA["hour_ending"], unit="h")
)

# rename to match your target format
hyattcs_shadow_price_DA = hyattcs_shadow_price_DA.rename(
    columns={
        "MONITORED_FACILITY_DESC": "nodename",
        "shadow_price": "shadow_price_DA"
    }
)

# reorder columns
hyattcs_shadow_price_DA = hyattcs_shadow_price_DA[
    ["datetime_ending_ept", "nodename", "shadow_price_DA"]
].sort_values(
    ["nodename", "datetime_ending_ept"]
).drop_duplicates(
    subset=["nodename", "datetime_ending_ept"],
    keep="last"
).reset_index(drop=True)

hyattcs_shadow_price_DA.head()

,datetime_ending_ept,nodename,shadow_price_DA
0,2026-04-16 07:00:00,HYATTCS_345KVMAR-HYA2_1_LN,8.71
1,2026-04-16 08:00:00,HYATTCS_345KVMAR-HYA2_1_LN,2.13
2,2026-04-16 11:00:00,HYATTCS_345KVMAR-HYA2_1_LN,17.95
3,2026-04-16 12:00:00,HYATTCS_345KVMAR-HYA2_1_LN,136.58
4,2026-04-16 13:00:00,HYATTCS_345KVMAR-HYA2_1_LN,276.75


In [148]:
hyattcs_shadow_price["shadow_price"].max()

np.float64(2000.0)

## county

In [149]:
# Get all nodenames that belong to agg_nodename == "AEP" in hourly_load
aep_nodes = (
    hourly_load.loc[hourly_load["agg_nodename"].isin(["AEP", "DAY", "ATSI"]), "nodename"]
    .dropna()
    .unique()
)

# Map those AEP nodenames to county_name using node_cty_map
aep_node_counties = (
    node_cty_map.loc[node_cty_map["nodename"].isin(aep_nodes), ["nodename", "county_name"]]
    .dropna(subset=["county_name"])
    .drop_duplicates()
)

# Get all county_name touched by those AEP nodenames
aep_counties = aep_node_counties["county_name"].unique()

# From those counties, get all nodenames in node_cty_map
all_nodes_in_aep_counties = (
    node_cty_map.loc[node_cty_map["county_name"].isin(aep_counties), "nodename"]
    .dropna()
    .unique()
)

# Select all rows from hourly_load whose nodename is in those nodenames
hourly_load_aep_counties = hourly_load.loc[
    hourly_load["nodename"].isin(all_nodes_in_aep_counties)
].copy()

In [150]:
hourly_load_county = hourly_load_aep_counties[["datetime_ending_ept", "agg_nodename", "nodename", "distribution_factor", "forecast_area", "forecast_load_mw", "distributed_load_mw"]].merge(node_cty_map[["nodename", "county_name", "latitude", "longitude", "state_short_name"]], on="nodename", how="left")

county_load = (
    hourly_load_county
    .groupby(
        ["datetime_ending_ept", "county_name", "state_short_name"],
        as_index=False
    )
    .agg(
        distributed_load_mw=("distributed_load_mw", "sum"),
        latitude=("latitude", "mean"),
        longitude=("longitude", "mean")
    )
    .sort_values(["datetime_ending_ept", "county_name"])
)

In [151]:
county_geo = (
    county_load
    .groupby("county_name", as_index=False)
    .agg(
        latitude=("latitude", "mean"),
        longitude=("longitude", "mean")
    )
)

# round to 2 decimals
county_geo["latitude"] = county_geo["latitude"].round(2)
county_geo["longitude"] = county_geo["longitude"].round(2)

county_load = county_load[["datetime_ending_ept", "county_name", "state_short_name", "distributed_load_mw"]].merge(county_geo, on="county_name", how="left").copy()

In [152]:
county_load = county_load[county_load["state_short_name"] == "OH"].copy()

county_load = county_load[(county_load["datetime_ending_ept"] >= "2026-04-12")].copy()

In [153]:
county_load["datetime_ending_ept"] = pd.to_datetime(county_load["datetime_ending_ept"])

county_load_shadow_price = county_load.merge(
    hyattcs_shadow_price[["datetime_ending_ept", "shadow_price"]],
    on=["datetime_ending_ept"],
    how="left"
)

county_load_shadow_price["shadow_price"] = county_load_shadow_price["shadow_price"].fillna(0)

In [154]:
# row-level filter
df_filtered = county_load_shadow_price[
    (county_load_shadow_price["shadow_price"].notna()) &
    (county_load_shadow_price["shadow_price"] != 0)
].copy()

# keep only counties with mean load > 20
county_load_shadow = df_filtered[
    df_filtered.groupby("county_name")["distributed_load_mw"].transform("mean") > 20
].copy()

### plot

In [155]:
county_load_shadow_price = county_load_shadow_price.merge(
    hyattcs_shadow_price_DA[["datetime_ending_ept", "shadow_price_DA"]],
    on=["datetime_ending_ept"],
    how="left"
)

county_load_shadow_price["shadow_price_DA"] = county_load_shadow_price["shadow_price_DA"].fillna(0)



In [156]:
df = county_load_shadow_price.copy()

node = "OH_Delaware"

sub = df[df["county_name"] == node].copy()

# ensure datetime is sorted
sub["datetime_ending_ept"] = pd.to_datetime(sub["datetime_ending_ept"])
sub = sub.sort_values("datetime_ending_ept")

fig = go.Figure()

# Left y-axis: distributed load
fig.add_trace(
    go.Scatter(
        x=sub["datetime_ending_ept"],
        y=sub["distributed_load_mw"],
        name="Distributed Load (MW)",
        line=dict(color="blue"),
        yaxis="y1"
    )
)

# Right y-axis: shadow price
fig.add_trace(
    go.Scatter(
        x=sub["datetime_ending_ept"],
        y=sub["shadow_price"],
        name="Shadow Price",
        line=dict(color="red"),
        yaxis="y2"
    )
)


fig.add_trace(
    go.Scatter(
        x=sub["datetime_ending_ept"],
        y=sub["shadow_price_DA"],
        name="Shadow Price DA",
        line=dict(color="green"),
        yaxis="y2"
    )
)

# Layout with dual axis
fig.update_layout(
    title=f"{node}: Load vs Shadow Price",
    xaxis=dict(title="Datetime"),
    
    yaxis=dict(
        title="Distributed Load (MW)",
        side="left"
    ),
    
    yaxis2=dict(
        title="Shadow Price",
        overlaying="y",
        side="right"
    ),
    
    legend=dict(x=0.01, y=0.99),
    height=500,
    width=2000
)

fig.show()

# FREMNT

## shadow

In [34]:
# copy
df = RT_constr[RT_constr["MONITORED_FACILITY_DESC"].str.contains("FREMNT-FREMONTC")].copy()

# hourly columns
hour_cols = [str(i) for i in range(1, 25)]

# keep only rows that have at least one hourly value
df = df.dropna(subset=hour_cols, how="all")

df["file_date"] = pd.to_datetime(df["file_date"])

# reshape to long format
fremnt_shadow_price = df.melt(
    id_vars=["MONITORED_FACILITY_DESC", "file_date"],
    value_vars=hour_cols,
    var_name="hour_ending",
    value_name="shadow_price"
)

# drop null hourly values
fremnt_shadow_price = fremnt_shadow_price.dropna(
    subset=["shadow_price"]
).copy()

# convert HE to int
fremnt_shadow_price["hour_ending"] = (
    fremnt_shadow_price["hour_ending"].astype(int)
)

# build datetime_ending_ept
fremnt_shadow_price["datetime_ending_ept"] = (
    fremnt_shadow_price["file_date"]
    + pd.to_timedelta(
        fremnt_shadow_price["hour_ending"],
        unit="h"
    )
)

# rename to match your target format
fremnt_shadow_price = fremnt_shadow_price.rename(
    columns={"MONITORED_FACILITY_DESC": "nodename"}
)

# reorder columns
fremnt_shadow_price = fremnt_shadow_price[
    ["datetime_ending_ept", "nodename", "shadow_price"]
].sort_values(
    ["nodename", "datetime_ending_ept"]
).drop_duplicates(
    subset=["nodename", "datetime_ending_ept"],
    keep="last"
).reset_index(drop=True)

fremnt_shadow_price.head()

,datetime_ending_ept,nodename,shadow_price
0,2026-04-02 19:00:00,FREMNT-FREMONTC A 138 KV,166.67
1,2026-04-02 20:00:00,FREMNT-FREMONTC A 138 KV,1210.65
2,2026-04-02 21:00:00,FREMNT-FREMONTC A 138 KV,1322.57
3,2026-04-02 22:00:00,FREMNT-FREMONTC A 138 KV,538.15
4,2026-04-02 23:00:00,FREMNT-FREMONTC A 138 KV,90.84


In [35]:
# copy
df = DA_constr[
    DA_constr["MONITORED_FACILITY_DESC"].str.contains("FREMNT")
].copy()

# hourly columns
hour_cols = [str(i) for i in range(1, 25)]

# keep only rows that have at least one hourly value
df = df.dropna(subset=hour_cols, how="all")

df["file_date"] = pd.to_datetime(df["file_date"])

# reshape to long format
fremnt_shadow_price_DA = df.melt(
    id_vars=["MONITORED_FACILITY_DESC", "file_date"],
    value_vars=hour_cols,
    var_name="hour_ending",
    value_name="shadow_price"
)

# drop null hourly values
fremnt_shadow_price_DA = fremnt_shadow_price_DA.dropna(
    subset=["shadow_price"]
).copy()

# convert HE to int
fremnt_shadow_price_DA["hour_ending"] = (
    fremnt_shadow_price_DA["hour_ending"].astype(int)
)

# build datetime_ending_ept
fremnt_shadow_price_DA["datetime_ending_ept"] = (
    fremnt_shadow_price_DA["file_date"]
    + pd.to_timedelta(
        fremnt_shadow_price_DA["hour_ending"],
        unit="h"
    )
)

# rename to match your target format
fremnt_shadow_price_DA = fremnt_shadow_price_DA.rename(
    columns={
        "MONITORED_FACILITY_DESC": "nodename",
        "shadow_price": "shadow_price_DA"
    }
)

# reorder columns
fremnt_shadow_price_DA = fremnt_shadow_price_DA[
    ["datetime_ending_ept", "nodename", "shadow_price_DA"]
].sort_values(
    ["nodename", "datetime_ending_ept"]
).drop_duplicates(
    subset=["nodename", "datetime_ending_ept"],
    keep="last"
).reset_index(drop=True)

fremnt_shadow_price_DA.head()

,datetime_ending_ept,nodename,shadow_price_DA
0,2026-04-01 07:00:00,FREMNT_138KVFRE-FRE1_1_LN,54.03
1,2026-04-01 09:00:00,FREMNT_138KVFRE-FRE1_1_LN,12.51
2,2026-04-01 14:00:00,FREMNT_138KVFRE-FRE1_1_LN,48.43
3,2026-04-01 16:00:00,FREMNT_138KVFRE-FRE1_1_LN,123.44
4,2026-04-01 19:00:00,FREMNT_138KVFRE-FRE1_1_LN,54.87


In [36]:
fremnt_shadow_price["shadow_price"].max()

np.float64(2000.0)

## county

In [37]:
# Get all nodenames that belong to agg_nodename == "ATSI" in hourly_load
atsi_nodes = (
    hourly_load.loc[hourly_load["agg_nodename"].isin(["ATSI"]), "nodename"]
    .dropna()
    .unique()
)

# Map those ATSI nodenames to county_name using node_cty_map
atsi_node_counties = (
    node_cty_map.loc[node_cty_map["nodename"].isin(atsi_nodes), ["nodename", "county_name"]]
    .dropna(subset=["county_name"])
    .drop_duplicates()
)

# Get all county_name touched by those ATSI nodenames
atsi_counties = atsi_node_counties["county_name"].unique()

# From those counties, get all nodenames in node_cty_map
all_nodes_in_atsi_counties = (
    node_cty_map.loc[node_cty_map["county_name"].isin(atsi_counties), "nodename"]
    .dropna()
    .unique()
)

# Select all rows from hourly_load whose nodename is in those nodenames
hourly_load_atsi_counties = hourly_load.loc[
    hourly_load["nodename"].isin(all_nodes_in_atsi_counties)
].copy()

In [38]:
hourly_load_county = hourly_load_atsi_counties[["datetime_ending_ept", "agg_nodename", "nodename", "distribution_factor", "forecast_area", "forecast_load_mw", "distributed_load_mw"]].merge(node_cty_map[["nodename", "county_name", "latitude", "longitude", "state_short_name"]], on="nodename", how="left")

county_load = (
    hourly_load_county
    .groupby(
        ["datetime_ending_ept", "county_name", "state_short_name"],
        as_index=False
    )
    .agg(
        distributed_load_mw=("distributed_load_mw", "sum"),
        latitude=("latitude", "mean"),
        longitude=("longitude", "mean")
    )
    .sort_values(["datetime_ending_ept", "county_name"])
)

In [39]:
county_geo = (
    county_load
    .groupby("county_name", as_index=False)
    .agg(
        latitude=("latitude", "mean"),
        longitude=("longitude", "mean")
    )
)

# round to 2 decimals
county_geo["latitude"] = county_geo["latitude"].round(2)
county_geo["longitude"] = county_geo["longitude"].round(2)

county_load = county_load[["datetime_ending_ept", "county_name", "state_short_name", "distributed_load_mw"]].merge(county_geo, on="county_name", how="left").copy()

In [40]:
county_load = county_load[county_load["state_short_name"].isin(["OH"])].copy()

county_load = county_load[(county_load["datetime_ending_ept"] >= "2026-05-01")].copy()

In [41]:
county_load["datetime_ending_ept"] = pd.to_datetime(county_load["datetime_ending_ept"])

county_load_shadow_price = county_load.merge(
    fremnt_shadow_price[["datetime_ending_ept", "shadow_price"]],
    on=["datetime_ending_ept"],
    how="left"
)

county_load_shadow_price["shadow_price"] = county_load_shadow_price["shadow_price"].fillna(0)

In [42]:
# row-level filter
df_filtered = county_load_shadow_price[
    (county_load_shadow_price["shadow_price"].notna()) &
    (county_load_shadow_price["shadow_price"] != 0)
].copy()

# keep only counties with max load > 20
county_load_shadow = df_filtered[
    df_filtered.groupby("county_name")["distributed_load_mw"].transform("max") > 20
].copy()

In [43]:
df = county_load_shadow.copy()

feature_cols = ["distributed_load_mw"] 

# 3. correlation between each feature and shadow_price for each nodename
corr_by_node = (
    df.groupby("county_name")
      .apply(lambda g: g[feature_cols].corrwith(g["shadow_price"]))
      .reset_index()
)

corr_by_node.sort_values(by="distributed_load_mw", ascending=False)

,county_name,distributed_load_mw
33,OH_Williams,0.355742
1,OH_Ashtabula,0.346683
5,OH_Crawford,0.341167
27,OH_Sandusky,0.340394
7,OH_Defiance,0.316589
3,OH_Clark,0.307028
32,OH_Wayne,0.300410
30,OH_Trumbull,0.296722
2,OH_Carroll,0.292721
4,OH_Columbiana,0.288426


### plot

In [44]:
county_load_shadow_price = county_load_shadow_price.merge(
    fremnt_shadow_price_DA[["datetime_ending_ept", "shadow_price_DA"]],
    on=["datetime_ending_ept"],
    how="left"
)

county_load_shadow_price["shadow_price_DA"] = county_load_shadow_price["shadow_price_DA"].fillna(0)

In [45]:
df = county_load_shadow_price.copy()

node = "OH_Henry"

sub = df[df["county_name"] == node].copy()

# ensure datetime is sorted
sub["datetime_ending_ept"] = pd.to_datetime(sub["datetime_ending_ept"])
sub = sub.sort_values("datetime_ending_ept")

fig = go.Figure()

# Left y-axis: distributed load
fig.add_trace(
    go.Scatter(
        x=sub["datetime_ending_ept"],
        y=sub["distributed_load_mw"],
        name="Distributed Load (MW)",
        line=dict(color="blue"),
        yaxis="y1"
    )
)

# Right y-axis: shadow price
fig.add_trace(
    go.Scatter(
        x=sub["datetime_ending_ept"],
        y=sub["shadow_price"],
        name="Shadow Price",
        line=dict(color="red"),
        yaxis="y2"
    )
)

fig.add_trace(
    go.Scatter(
        x=sub["datetime_ending_ept"],
        y=sub["shadow_price_DA"],
        name="Shadow Price DA",
        line=dict(color="green"),
        yaxis="y2"
    )
)

# Layout with dual axis
fig.update_layout(
    title=f"{node}: Load vs Shadow Price",
    xaxis=dict(title="Datetime"),
    
    yaxis=dict(
        title="Distributed Load (MW)",
        side="left"
    ),
    
    yaxis2=dict(
        title="Shadow Price",
        overlaying="y",
        side="right"
    ),
    
    legend=dict(x=0.01, y=0.99),
    height=500,
    width=2000
)

fig.show()

# Batesville

## shadow

In [135]:
# copy
df = RT_constr[RT_constr["MONITORED_FACILITY_DESC"] == "Batesville-Hubble 138 l/o Tanners Crk-Miami Fort 345"].copy()

# hourly columns
hour_cols = [str(i) for i in range(1, 25)]

# keep only rows that have at least one hourly value
df = df.dropna(subset=hour_cols, how="all")

df["file_date"] = pd.to_datetime(df["file_date"])

# reshape to long format
batesville_shadow_price = df.melt(
    id_vars=["MONITORED_FACILITY_DESC", "file_date"],
    value_vars=hour_cols,
    var_name="hour_ending",
    value_name="shadow_price"
)

# drop null hourly values
batesville_shadow_price = batesville_shadow_price.dropna(subset=["shadow_price"]).copy()

# convert HE to int
batesville_shadow_price["hour_ending"] = batesville_shadow_price["hour_ending"].astype(int)

# set operating date
# replace with the actual DA operating date if needed
operating_date = pd.Timestamp("2026-04-21")

# build datetime_ending_ept
batesville_shadow_price["datetime_ending_ept"] = (
    batesville_shadow_price["file_date"] + pd.to_timedelta(batesville_shadow_price["hour_ending"], unit="h")
)

# rename to match your target format
batesville_shadow_price = batesville_shadow_price.rename(
    columns={"MONITORED_FACILITY_DESC": "nodename"}
)

# reorder columns
batesville_shadow_price = batesville_shadow_price[
    ["datetime_ending_ept", "nodename", "shadow_price"]
].sort_values(["nodename", "datetime_ending_ept"]
).drop_duplicates(
    subset=["nodename", "datetime_ending_ept"],
    keep="last"
).reset_index(drop=True)

batesville_shadow_price.head()

,datetime_ending_ept,nodename,shadow_price
0,2026-04-05 11:00:00,Batesville-Hubble 138 l/o Tanners Crk-Miami Fo...,564.99
1,2026-04-05 12:00:00,Batesville-Hubble 138 l/o Tanners Crk-Miami Fo...,622.40
2,2026-04-05 13:00:00,Batesville-Hubble 138 l/o Tanners Crk-Miami Fo...,44.11
3,2026-04-05 14:00:00,Batesville-Hubble 138 l/o Tanners Crk-Miami Fo...,414.60
4,2026-04-05 15:00:00,Batesville-Hubble 138 l/o Tanners Crk-Miami Fo...,20.80


In [136]:
# copy
df = DA_constr[DA_constr["MONITORED_FACILITY_DESC"] == "Batesville-Hubble 138 l/o Tanners Crk-Miami Fort 345"].copy()

# hourly columns
hour_cols = [str(i) for i in range(1, 25)]

# keep only rows that have at least one hourly value
df = df.dropna(subset=hour_cols, how="all")

df["file_date"] = pd.to_datetime(df["file_date"])

# reshape to long format
batesville_shadow_price_DA = df.melt(
    id_vars=["MONITORED_FACILITY_DESC", "file_date"],
    value_vars=hour_cols,
    var_name="hour_ending",
    value_name="shadow_price"
)

# drop null hourly values
batesville_shadow_price_DA = batesville_shadow_price_DA.dropna(subset=["shadow_price"]).copy()

# convert HE to int
batesville_shadow_price_DA["hour_ending"] = batesville_shadow_price_DA["hour_ending"].astype(int)

# build datetime_ending_ept
batesville_shadow_price_DA["datetime_ending_ept"] = (
    batesville_shadow_price_DA["file_date"] + 
    pd.to_timedelta(batesville_shadow_price_DA["hour_ending"], unit="h")
)

# rename to match your target format
batesville_shadow_price_DA = batesville_shadow_price_DA.rename(
    columns={
        "MONITORED_FACILITY_DESC": "nodename",
        "shadow_price": "shadow_price_DA"
    }
)

# reorder columns
batesville_shadow_price_DA = batesville_shadow_price_DA[
    ["datetime_ending_ept", "nodename", "shadow_price_DA"]
].sort_values(
    ["nodename", "datetime_ending_ept"]
).drop_duplicates(
    subset=["nodename", "datetime_ending_ept"],
    keep="last"
).reset_index(drop=True)

batesville_shadow_price_DA.head()

,datetime_ending_ept,nodename,shadow_price_DA
0,2026-04-08 07:00:00,Batesville-Hubble 138 l/o Tanners Crk-Miami Fo...,63.74
1,2026-04-08 20:00:00,Batesville-Hubble 138 l/o Tanners Crk-Miami Fo...,10.10
2,2026-04-08 21:00:00,Batesville-Hubble 138 l/o Tanners Crk-Miami Fo...,46.68
3,2026-04-14 14:00:00,Batesville-Hubble 138 l/o Tanners Crk-Miami Fo...,34.19
4,2026-04-14 15:00:00,Batesville-Hubble 138 l/o Tanners Crk-Miami Fo...,28.31


In [137]:
batesville_shadow_price["shadow_price"].max()

np.float64(2000.0)

## county

In [138]:
# Get all nodenames that belong to agg_nodename == "DEOK" in hourly_load
deok_nodes = (
    hourly_load.loc[hourly_load["agg_nodename"] == "DEOK", "nodename"]
    .dropna()
    .unique()
)

# Map those DEOK nodenames to county_name using node_cty_map
deok_node_counties = (
    node_cty_map.loc[node_cty_map["nodename"].isin(deok_nodes), ["nodename", "county_name"]]
    .dropna(subset=["county_name"])
    .drop_duplicates()
)

# Get all county_name touched by those DEOK nodenames
deok_counties = deok_node_counties["county_name"].unique()

# From those counties, get all nodenames in node_cty_map
all_nodes_in_deok_counties = (
    node_cty_map.loc[node_cty_map["county_name"].isin(deok_counties), "nodename"]
    .dropna()
    .unique()
)

# Select all rows from hourly_load whose nodename is in those nodenames
hourly_load_deok_counties = hourly_load.loc[
    hourly_load["nodename"].isin(all_nodes_in_deok_counties)
].copy()

In [139]:
hourly_load_county = hourly_load_deok_counties[["datetime_ending_ept", "agg_nodename", "nodename", "distribution_factor", "forecast_area", "forecast_load_mw", "distributed_load_mw"]].merge(node_cty_map[["nodename", "county_name", "latitude", "longitude", "state_short_name"]], on="nodename", how="left")

county_load = (
    hourly_load_county
    .groupby(
        ["datetime_ending_ept", "county_name", "state_short_name"],
        as_index=False
    )
    .agg(
        distributed_load_mw=("distributed_load_mw", "sum"),
        latitude=("latitude", "mean"),
        longitude=("longitude", "mean")
    )
    .sort_values(["datetime_ending_ept", "county_name"])
)

In [140]:
county_geo = (
    county_load
    .groupby("county_name", as_index=False)
    .agg(
        latitude=("latitude", "mean"),
        longitude=("longitude", "mean")
    )
)

# round to 2 decimals
county_geo["latitude"] = county_geo["latitude"].round(2)
county_geo["longitude"] = county_geo["longitude"].round(2)

county_load = county_load[["datetime_ending_ept", "county_name", "state_short_name", "distributed_load_mw"]].merge(county_geo, on="county_name", how="left").copy()

In [141]:
county_load = county_load[(county_load["datetime_ending_ept"] >= "2026-04-05")].copy()

In [142]:
county_load["datetime_ending_ept"] = pd.to_datetime(county_load["datetime_ending_ept"])

county_load_shadow_price = county_load.merge(
    batesville_shadow_price[["datetime_ending_ept", "shadow_price"]],
    on=["datetime_ending_ept"],
    how="left"
)

county_load_shadow_price["shadow_price"] = county_load_shadow_price["shadow_price"].fillna(0)

county_load_shadow = county_load_shadow_price[
    (county_load_shadow_price["shadow_price"].notna()) &
    (county_load_shadow_price["shadow_price"] != 0)
].copy()

In [143]:
df = county_load_shadow.copy()

feature_cols = ["distributed_load_mw"] 

# correlation between each feature and shadow_price for each nodename
corr_by_node = (
    df.groupby("county_name")
      .apply(lambda g: g[feature_cols].corrwith(g["shadow_price"]))
      .reset_index()
)

corr_by_node.sort_values(by="distributed_load_mw", ascending=False)

,county_name,distributed_load_mw
7,OH_Clermont,0.459677
2,KY_Grant,0.380682
1,KY_Campbell,0.366471
6,OH_Butler,0.354518
9,OH_Warren,0.339081
8,OH_Hamilton,0.294070
0,KY_Boone,0.291723
3,KY_Kenton,0.269686
5,OH_Brown,0.063633
4,KY_Pendleton,0.030465


### plot

In [144]:
county_load_shadow_price = county_load_shadow_price.merge(
    batesville_shadow_price_DA[["datetime_ending_ept", "shadow_price_DA"]],
    on=["datetime_ending_ept"],
    how="left"
)

county_load_shadow_price["shadow_price_DA"] = county_load_shadow_price["shadow_price_DA"].fillna(0)



In [145]:

df = county_load_shadow_price.copy()

node = "OH_Butler"

sub = df[df["county_name"] == node].copy()

# ensure datetime is sorted
sub["datetime_ending_ept"] = pd.to_datetime(sub["datetime_ending_ept"])
sub = sub.sort_values("datetime_ending_ept")

fig = go.Figure()

# Left y-axis: distributed load
fig.add_trace(
    go.Scatter(
        x=sub["datetime_ending_ept"],
        y=sub["distributed_load_mw"],
        name="Distributed Load (MW)",
        line=dict(color="blue"),
        yaxis="y1"
    )
)

# Right y-axis: shadow price
fig.add_trace(
    go.Scatter(
        x=sub["datetime_ending_ept"],
        y=sub["shadow_price"],
        name="Shadow Price",
        line=dict(color="red"),
        yaxis="y2"
    )
)

fig.add_trace(
    go.Scatter(
        x=sub["datetime_ending_ept"],
        y=sub["shadow_price_DA"],
        name="Shadow Price DA",
        line=dict(color="green"),
        yaxis="y2"
    )
)

# Layout with dual axis
fig.update_layout(
    title=f"{node}: Load vs Shadow Price",
    xaxis=dict(title="Datetime"),
    
    yaxis=dict(
        title="Distributed Load (MW)",
        side="left"
    ),
    
    yaxis2=dict(
        title="Shadow Price",
        overlaying="y",
        side="right"
    ),
    
    legend=dict(x=0.01, y=0.99),
    height=500,
    width=2000
)

fig.show()

# NILES

## shadow

In [123]:
# copy
df = RT_constr[RT_constr["MONITORED_FACILITY_DESC"].str.contains("NILES")].copy()

# hourly columns
hour_cols = [str(i) for i in range(1, 25)]

# keep only rows that have at least one hourly value
df = df.dropna(subset=hour_cols, how="all")

df["file_date"] = pd.to_datetime(df["file_date"])

# reshape to long format
niles_shadow_price = df.melt(
    id_vars=["MONITORED_FACILITY_DESC", "file_date"],
    value_vars=hour_cols,
    var_name="hour_ending",
    value_name="shadow_price"
)

# drop null hourly values
niles_shadow_price = niles_shadow_price.dropna(subset=["shadow_price"]).copy()

# convert HE to int
niles_shadow_price["hour_ending"] = niles_shadow_price["hour_ending"].astype(int)

# build datetime_ending_ept
niles_shadow_price["datetime_ending_ept"] = (
    niles_shadow_price["file_date"] + pd.to_timedelta(niles_shadow_price["hour_ending"], unit="h")
)

# rename to match your target format
niles_shadow_price = niles_shadow_price.rename(
    columns={"MONITORED_FACILITY_DESC": "nodename"}
)

# reorder columns
niles_shadow_price = niles_shadow_price[
    ["datetime_ending_ept", "nodename", "shadow_price"]
].sort_values(["nodename", "datetime_ending_ept"]
).drop_duplicates(
    subset=["nodename", "datetime_ending_ept"],
    keep="last"
).reset_index(drop=True)

niles_shadow_price.head()

,datetime_ending_ept,nodename,shadow_price
0,2026-04-28 10:00:00,NILES-EVERGFE A 138 KV,901.95
1,2026-04-28 11:00:00,NILES-EVERGFE A 138 KV,203.16
2,2026-04-23 23:00:00,NILES-EVERGFE B 138 KV,115.65
3,2026-04-24 00:00:00,NILES-EVERGFE B 138 KV,240.29
4,2026-04-24 01:00:00,NILES-EVERGFE B 138 KV,65.95


In [124]:
# copy
df = DA_constr[DA_constr["MONITORED_FACILITY_DESC"].str.contains("NILES")].copy()

# hourly columns
hour_cols = [str(i) for i in range(1, 25)]

# keep only rows that have at least one hourly value
df = df.dropna(subset=hour_cols, how="all")

df["file_date"] = pd.to_datetime(df["file_date"])

# reshape to long format
niles_shadow_price_DA = df.melt(
    id_vars=["MONITORED_FACILITY_DESC", "file_date"],
    value_vars=hour_cols,
    var_name="hour_ending",
    value_name="shadow_price"
)

# drop null hourly values
niles_shadow_price_DA = niles_shadow_price_DA.dropna(subset=["shadow_price"]).copy()

# convert HE to int
niles_shadow_price_DA["hour_ending"] = niles_shadow_price_DA["hour_ending"].astype(int)

# build datetime_ending_ept
niles_shadow_price_DA["datetime_ending_ept"] = (
    niles_shadow_price_DA["file_date"] + 
    pd.to_timedelta(niles_shadow_price_DA["hour_ending"], unit="h")
)

# rename to match your target format
niles_shadow_price_DA = niles_shadow_price_DA.rename(
    columns={
        "MONITORED_FACILITY_DESC": "nodename",
        "shadow_price": "shadow_price_DA"
    }
)

# reorder columns
niles_shadow_price_DA = niles_shadow_price_DA[
    ["datetime_ending_ept", "nodename", "shadow_price_DA"]
].sort_values(
    ["nodename", "datetime_ending_ept"]
).drop_duplicates(
    subset=["nodename", "datetime_ending_ept"],
    keep="last"
).reset_index(drop=True)

niles_shadow_price_DA.head()

,datetime_ending_ept,nodename,shadow_price_DA
0,2026-04-28 13:00:00,NILES_138KVNIL-EVE1_1_LN,68.74
1,2026-04-28 14:00:00,NILES_138KVNIL-EVE1_1_LN,60.60
2,2026-04-28 15:00:00,NILES_138KVNIL-EVE1_1_LN,48.36
3,2026-04-28 16:00:00,NILES_138KVNIL-EVE1_1_LN,42.86
4,2026-04-28 17:00:00,NILES_138KVNIL-EVE1_1_LN,123.81


In [125]:
niles_shadow_price["shadow_price"].max()

np.float64(1551.07)

## county

In [126]:
# Get all nodenames that belong to agg_nodename == "ATSI" in hourly_load
atsi_nodes = (
    hourly_load.loc[hourly_load["agg_nodename"].isin(["ATSI", "PENELEC"]), "nodename"]
    .dropna()
    .unique()
)

# Map those ATSI nodenames to county_name using node_cty_map
atsi_node_counties = (
    node_cty_map.loc[node_cty_map["nodename"].isin(atsi_nodes), ["nodename", "county_name"]]
    .dropna(subset=["county_name"])
    .drop_duplicates()
)

# Get all county_name touched by those ATSI nodenames
atsi_counties = atsi_node_counties["county_name"].unique()

# From those counties, get all nodenames in node_cty_map
all_nodes_in_atsi_counties = (
    node_cty_map.loc[node_cty_map["county_name"].isin(atsi_counties), "nodename"]
    .dropna()
    .unique()
)

# Select all rows from hourly_load whose nodename is in those nodenames
hourly_load_atsi_counties = hourly_load.loc[
    hourly_load["nodename"].isin(all_nodes_in_atsi_counties)
].copy()

In [127]:
hourly_load_county = hourly_load_atsi_counties[["datetime_ending_ept", "agg_nodename", "nodename", "distribution_factor", "forecast_area", "forecast_load_mw", "distributed_load_mw"]].merge(node_cty_map[["nodename", "county_name", "latitude", "longitude", "state_short_name"]], on="nodename", how="left")

county_load = (
    hourly_load_county
    .groupby(
        ["datetime_ending_ept", "county_name", "state_short_name"],
        as_index=False
    )
    .agg(
        distributed_load_mw=("distributed_load_mw", "sum"),
        latitude=("latitude", "mean"),
        longitude=("longitude", "mean")
    )
    .sort_values(["datetime_ending_ept", "county_name"])
)

In [128]:
county_geo = (
    county_load
    .groupby("county_name", as_index=False)
    .agg(
        latitude=("latitude", "mean"),
        longitude=("longitude", "mean")
    )
)

# round to 2 decimals
county_geo["latitude"] = county_geo["latitude"].round(2)
county_geo["longitude"] = county_geo["longitude"].round(2)

county_load = county_load[["datetime_ending_ept", "county_name", "state_short_name", "distributed_load_mw"]].merge(county_geo, on="county_name", how="left").copy()

In [129]:
county_load = county_load[county_load["state_short_name"].isin(["OH", "PA"])].copy()

county_load = county_load[(county_load["datetime_ending_ept"] >= "2026-04-20")].copy()

In [130]:
county_load["datetime_ending_ept"] = pd.to_datetime(county_load["datetime_ending_ept"])

county_load_shadow_price = county_load.merge(
    niles_shadow_price[["datetime_ending_ept", "shadow_price"]],
    on=["datetime_ending_ept"],
    how="left"
)

county_load_shadow_price["shadow_price"] = county_load_shadow_price["shadow_price"].fillna(0)

In [131]:
# row-level filter
df_filtered = county_load_shadow_price[
    (county_load_shadow_price["shadow_price"].notna()) &
    (county_load_shadow_price["shadow_price"] != 0)
].copy()

# keep only counties with max load > 20
county_load_shadow = df_filtered[
    df_filtered.groupby("county_name")["distributed_load_mw"].transform("max") > 20
].copy()

In [132]:
df = county_load_shadow.copy()

feature_cols = ["distributed_load_mw"] 

# 3. correlation between each feature and shadow_price for each nodename
corr_by_node = (
    df.groupby("county_name")
      .apply(lambda g: g[feature_cols].corrwith(g["shadow_price"]))
      .reset_index()
)

corr_by_node.sort_values(by="distributed_load_mw", ascending=False)

,county_name,distributed_load_mw
23,OH_Morrow,0.385678
44,PA_Clearfield,0.341376
41,PA_Cambria,0.320210
51,PA_Lawrence,0.276829
42,PA_Centre,0.274017
...,...,...
1,OH_Ashtabula,-0.086034
54,PA_Mifflin,-0.132622
35,PA_Armstrong,-0.133120
21,OH_Marion,-0.138932


### plot

In [133]:
county_load_shadow_price = county_load_shadow_price.merge(
    niles_shadow_price_DA[["datetime_ending_ept", "shadow_price_DA"]],
    on=["datetime_ending_ept"],
    how="left"
)

county_load_shadow_price["shadow_price_DA"] = county_load_shadow_price["shadow_price_DA"].fillna(0)

In [134]:
df = county_load_shadow_price.copy()

node = "OH_Trumbull"

sub = df[df["county_name"] == node].copy()

# ensure datetime is sorted
sub["datetime_ending_ept"] = pd.to_datetime(sub["datetime_ending_ept"])
sub = sub.sort_values("datetime_ending_ept")

fig = go.Figure()

# Left y-axis: distributed load
fig.add_trace(
    go.Scatter(
        x=sub["datetime_ending_ept"],
        y=sub["distributed_load_mw"],
        name="Distributed Load (MW)",
        line=dict(color="blue"),
        yaxis="y1"
    )
)

# Right y-axis: shadow price
fig.add_trace(
    go.Scatter(
        x=sub["datetime_ending_ept"],
        y=sub["shadow_price"],
        name="Shadow Price",
        line=dict(color="red"),
        yaxis="y2"
    )
)

fig.add_trace(
    go.Scatter(
        x=sub["datetime_ending_ept"],
        y=sub["shadow_price_DA"],
        name="Shadow Price DA",
        line=dict(color="green"),
        yaxis="y2"
    )
)

# Layout with dual axis
fig.update_layout(
    title=f"{node}: Load vs Shadow Price",
    xaxis=dict(title="Datetime"),
    
    yaxis=dict(
        title="Distributed Load (MW)",
        side="left"
    ),
    
    yaxis2=dict(
        title="Shadow Price",
        overlaying="y",
        side="right"
    ),
    
    legend=dict(x=0.01, y=0.99),
    height=500,
    width=2000
)

fig.show()

# PLEASNTV

## shadow

In [20]:
# copy
df = RT_constr[RT_constr["MONITORED_FACILITY_DESC"].str.contains("PLEASNTV-ASHBURN")].copy()

# hourly columns
hour_cols = [str(i) for i in range(1, 25)]

# keep only rows that have at least one hourly value
df = df.dropna(subset=hour_cols, how="all")

df["file_date"] = pd.to_datetime(df["file_date"])

# reshape to long format
pleasntv_shadow_price = df.melt(
    id_vars=["MONITORED_FACILITY_DESC", "file_date"],
    value_vars=hour_cols,
    var_name="hour_ending",
    value_name="shadow_price"
)

# drop null hourly values
pleasntv_shadow_price = pleasntv_shadow_price.dropna(subset=["shadow_price"]).copy()

# convert HE to int
pleasntv_shadow_price["hour_ending"] = pleasntv_shadow_price["hour_ending"].astype(int)

# build datetime_ending_ept
pleasntv_shadow_price["datetime_ending_ept"] = (
    pleasntv_shadow_price["file_date"] + pd.to_timedelta(
        pleasntv_shadow_price["hour_ending"], unit="h"
    )
)

# rename to match your target format
pleasntv_shadow_price = pleasntv_shadow_price.rename(
    columns={"MONITORED_FACILITY_DESC": "nodename"}
)

# reorder columns
pleasntv_shadow_price = pleasntv_shadow_price[
    ["datetime_ending_ept", "nodename", "shadow_price"]
].sort_values(
    ["nodename", "datetime_ending_ept"]
).drop_duplicates(
    subset=["nodename", "datetime_ending_ept"],
    keep="last"
).reset_index(drop=True)

pleasntv_shadow_price.head()

,datetime_ending_ept,nodename,shadow_price
0,2026-05-26 01:00:00,PLEASNTV-ASHBURN 274D A 230 KV,166.67
1,2026-05-26 06:00:00,PLEASNTV-ASHBURN 274D A 230 KV,167.02
2,2026-05-26 07:00:00,PLEASNTV-ASHBURN 274D A 230 KV,1532.39
3,2026-05-26 08:00:00,PLEASNTV-ASHBURN 274D A 230 KV,769.21
4,2026-05-26 09:00:00,PLEASNTV-ASHBURN 274D A 230 KV,1916.09


In [21]:
# copy
df = DA_constr[DA_constr["MONITORED_FACILITY_DESC"].str.contains("PLEASNTV-ASHBURN")].copy()

# hourly columns
hour_cols = [str(i) for i in range(1, 25)]

# keep only rows that have at least one hourly value
df = df.dropna(subset=hour_cols, how="all")

df["file_date"] = pd.to_datetime(df["file_date"])

# reshape to long format
pleasntv_shadow_price_DA = df.melt(
    id_vars=["MONITORED_FACILITY_DESC", "file_date"],
    value_vars=hour_cols,
    var_name="hour_ending",
    value_name="shadow_price"
)

# drop null hourly values
pleasntv_shadow_price_DA = pleasntv_shadow_price_DA.dropna(subset=["shadow_price"]).copy()

# convert HE to int
pleasntv_shadow_price_DA["hour_ending"] = pleasntv_shadow_price_DA["hour_ending"].astype(int)

# build datetime_ending_ept
pleasntv_shadow_price_DA["datetime_ending_ept"] = (
    pleasntv_shadow_price_DA["file_date"] + 
    pd.to_timedelta(pleasntv_shadow_price_DA["hour_ending"], unit="h")
)

# rename to match your target format
pleasntv_shadow_price_DA = pleasntv_shadow_price_DA.rename(
    columns={
        "MONITORED_FACILITY_DESC": "nodename",
        "shadow_price": "shadow_price_DA"
    }
)

# reorder columns
pleasntv_shadow_price_DA = pleasntv_shadow_price_DA[
    ["datetime_ending_ept", "nodename", "shadow_price_DA"]
].sort_values(
    ["nodename", "datetime_ending_ept"]
).drop_duplicates(
    subset=["nodename", "datetime_ending_ept"],
    keep="last"
).reset_index(drop=True)

pleasntv_shadow_price_DA.head()

,datetime_ending_ept,nodename,shadow_price_DA
0,2026-04-01 14:00:00,PLEASNTV-ASHBURN 274D A 230 KV,36.60
1,2026-04-01 15:00:00,PLEASNTV-ASHBURN 274D A 230 KV,50.13
2,2026-04-01 16:00:00,PLEASNTV-ASHBURN 274D A 230 KV,106.78
3,2026-04-01 17:00:00,PLEASNTV-ASHBURN 274D A 230 KV,122.38
4,2026-04-01 18:00:00,PLEASNTV-ASHBURN 274D A 230 KV,136.24


In [22]:
pleasntv_shadow_price["shadow_price"].max()

np.float64(2009.55)

## county

In [23]:
# Get all nodenames that belong to agg_nodename == "DOM" in hourly_load
dom_nodes = (
    hourly_load.loc[hourly_load["agg_nodename"] == "DOM", "nodename"]
    .dropna()
    .unique()
)

# Map those DOM nodenames to county_name using node_cty_map
dom_node_counties = (
    node_cty_map.loc[node_cty_map["nodename"].isin(dom_nodes), ["nodename", "county_name"]]
    .dropna(subset=["county_name"])
    .drop_duplicates()
)

# Get all county_name touched by those DOM nodenames
dom_counties = dom_node_counties["county_name"].unique()

# From those counties, get all nodenames in node_cty_map
all_nodes_in_dom_counties = (
    node_cty_map.loc[node_cty_map["county_name"].isin(dom_counties), "nodename"]
    .dropna()
    .unique()
)

# Select all rows from hourly_load whose nodename is in those nodenames
hourly_load_dom_counties = hourly_load.loc[
    hourly_load["nodename"].isin(all_nodes_in_dom_counties)
].copy()

In [24]:
hourly_load_county = hourly_load_dom_counties[["datetime_ending_ept", "agg_nodename", "nodename", "distribution_factor", "forecast_area", "forecast_load_mw", "distributed_load_mw"]].merge(node_cty_map[["nodename", "county_name", "latitude", "longitude", "state_short_name"]], on="nodename", how="left")

county_load = (
    hourly_load_county
    .groupby(
        ["datetime_ending_ept", "county_name", "state_short_name"],
        as_index=False
    )
    .agg(
        distributed_load_mw=("distributed_load_mw", "sum"),
        latitude=("latitude", "mean"),
        longitude=("longitude", "mean")
    )
    .sort_values(["datetime_ending_ept", "county_name"])
)

In [25]:
county_geo = (
    county_load
    .groupby("county_name", as_index=False)
    .agg(
        latitude=("latitude", "mean"),
        longitude=("longitude", "mean")
    )
)

# round to 2 decimals
county_geo["latitude"] = county_geo["latitude"].round(2)
county_geo["longitude"] = county_geo["longitude"].round(2)

county_load = county_load[["datetime_ending_ept", "county_name", "state_short_name", "distributed_load_mw"]].merge(county_geo, on="county_name", how="left").copy()

In [26]:
county_load = county_load[(county_load["datetime_ending_ept"] >= "2026-05-01")].copy()

In [27]:
county_load["datetime_ending_ept"] = pd.to_datetime(county_load["datetime_ending_ept"])

county_load_shadow_price = county_load.merge(
    pleasntv_shadow_price[["datetime_ending_ept", "shadow_price"]],
    on=["datetime_ending_ept"],
    how="left"
)

county_load_shadow_price["shadow_price"] = county_load_shadow_price["shadow_price"].fillna(0)

In [28]:
county_load_shadow_price.head()

,datetime_ending_ept,county_name,state_short_name,distributed_load_mw,latitude,longitude,shadow_price
0,2026-05-01,DC_District of Columbia,DC,61.58026,38.89,-77.01,0.0
1,2026-05-01,MD_Montgomery,MD,1213.32219,39.07,-77.21,0.0
2,2026-05-01,NC_Beaufort,NC,14.06267,35.60,-76.88,0.0
3,2026-05-01,NC_Bertie,NC,24.72181,36.01,-76.96,0.0
4,2026-05-01,NC_Chowan,NC,0.46172,36.05,-76.48,0.0


In [29]:
# row-level filter
df_filtered = county_load_shadow_price[
    (county_load_shadow_price["shadow_price"].notna()) &
    (county_load_shadow_price["shadow_price"] != 0)
].copy()

# keep only counties with mean load > 20
county_load_shadow = df_filtered[
    df_filtered.groupby("county_name")["distributed_load_mw"].transform("max") > 20
].copy()

In [30]:
df = county_load_shadow.copy()

feature_cols = ["distributed_load_mw"] 

# 3. correlation between each feature and shadow_price for each nodename
corr_by_node = (
    df.groupby("county_name")
      .apply(lambda g: g[feature_cols].corrwith(g["shadow_price"]))
      .reset_index()
)

corr_by_node.sort_values(by="distributed_load_mw", ascending=False)

,county_name,distributed_load_mw
84,VA_Waynesboro,0.587804
19,VA_Arlington,0.586414
16,VA_Alexandria,0.553476
51,VA_King George,0.548742
77,VA_Southampton,0.547959
...,...,...
9,NC_Hertford,-0.270124
14,NC_Perquimans,-0.290096
10,NC_Martin,-0.292958
4,NC_Bertie,-0.336285


In [31]:
corr_by_node[corr_by_node["county_name"] == "VA_Fairfax"]

,county_name,distributed_load_mw
35,VA_Fairfax,0.538828


### plot

In [32]:
county_load_shadow_price = county_load_shadow_price.merge(
    pleasntv_shadow_price_DA[["datetime_ending_ept", "shadow_price_DA"]],
    on=["datetime_ending_ept"],
    how="left"
)

county_load_shadow_price["shadow_price_DA"] = county_load_shadow_price["shadow_price_DA"].fillna(0)

In [33]:
df = county_load_shadow_price.copy()

node = "VA_Fairfax"

sub = df[df["county_name"] == node].copy()

# ensure datetime is sorted
sub["datetime_ending_ept"] = pd.to_datetime(sub["datetime_ending_ept"])
sub = sub.sort_values("datetime_ending_ept")

fig = go.Figure()

# Left y-axis: distributed load
fig.add_trace(
    go.Scatter(
        x=sub["datetime_ending_ept"],
        y=sub["distributed_load_mw"],
        name="Distributed Load (MW)",
        line=dict(color="blue"),
        yaxis="y1"
    )
)

# Right y-axis: shadow price
fig.add_trace(
    go.Scatter(
        x=sub["datetime_ending_ept"],
        y=sub["shadow_price"],
        name="Shadow Price",
        line=dict(color="red"),
        yaxis="y2"
    )
)

fig.add_trace(
    go.Scatter(
        x=sub["datetime_ending_ept"],
        y=sub["shadow_price_DA"],
        name="Shadow Price DA",
        line=dict(color="green"),
        yaxis="y2"
    )
)

# Layout with dual axis
fig.update_layout(
    title=f"{node}: Load vs Shadow Price",
    xaxis=dict(title="Datetime"),
    
    yaxis=dict(
        title="Distributed Load (MW)",
        side="left"
    ),
    
    yaxis2=dict(
        title="Shadow Price",
        overlaying="y",
        side="right"
    ),
    
    legend=dict(x=0.01, y=0.99),
    height=500,
    width=2000
)

fig.show()

# PLYWOOD

## shadow

In [46]:
# copy
df = RT_constr[
    RT_constr["MONITORED_FACILITY_DESC"].str.contains("PLYWOOD")
].copy()

# hourly columns
hour_cols = [str(i) for i in range(1, 25)]

# keep only rows that have at least one hourly value
df = df.dropna(subset=hour_cols, how="all")

df["file_date"] = pd.to_datetime(df["file_date"])

# reshape to long format
plywood_shadow_price = df.melt(
    id_vars=["MONITORED_FACILITY_DESC", "file_date"],
    value_vars=hour_cols,
    var_name="hour_ending",
    value_name="shadow_price"
)

# drop null hourly values
plywood_shadow_price = plywood_shadow_price.dropna(
    subset=["shadow_price"]
).copy()

# convert HE to int
plywood_shadow_price["hour_ending"] = (
    plywood_shadow_price["hour_ending"].astype(int)
)

# build datetime_ending_ept
plywood_shadow_price["datetime_ending_ept"] = (
    plywood_shadow_price["file_date"]
    + pd.to_timedelta(
        plywood_shadow_price["hour_ending"],
        unit="h"
    )
)

# rename to match your target format
plywood_shadow_price = plywood_shadow_price.rename(
    columns={"MONITORED_FACILITY_DESC": "nodename"}
)

# reorder columns
plywood_shadow_price = plywood_shadow_price[
    ["datetime_ending_ept", "nodename", "shadow_price"]
].sort_values(
    ["nodename", "datetime_ending_ept"]
).drop_duplicates(
    subset=["nodename", "datetime_ending_ept"],
    keep="last"
).reset_index(drop=True)

plywood_shadow_price.head()

,datetime_ending_ept,nodename,shadow_price
0,2026-04-12 20:00:00,PLYWOOD-SEDGEHIL 151A B 115 KV,666.67
1,2026-04-12 21:00:00,PLYWOOD-SEDGEHIL 151A B 115 KV,500.00
2,2026-04-13 20:00:00,PLYWOOD-SEDGEHIL 151A B 115 KV,1148.46
3,2026-04-13 21:00:00,PLYWOOD-SEDGEHIL 151A B 115 KV,879.52
4,2026-04-14 19:00:00,PLYWOOD-SEDGEHIL 151A B 115 KV,1000.00


In [47]:
# copy
df = DA_constr[
    DA_constr["MONITORED_FACILITY_DESC"].str.contains("PLYWOOD")
].copy()

# hourly columns
hour_cols = [str(i) for i in range(1, 25)]

# keep only rows that have at least one hourly value
df = df.dropna(subset=hour_cols, how="all")

df["file_date"] = pd.to_datetime(df["file_date"])

# reshape to long format
plywood_shadow_price_DA = df.melt(
    id_vars=["MONITORED_FACILITY_DESC", "file_date"],
    value_vars=hour_cols,
    var_name="hour_ending",
    value_name="shadow_price"
)

# drop null hourly values
plywood_shadow_price_DA = plywood_shadow_price_DA.dropna(
    subset=["shadow_price"]
).copy()

# convert HE to int
plywood_shadow_price_DA["hour_ending"] = (
    plywood_shadow_price_DA["hour_ending"].astype(int)
)

# build datetime_ending_ept
plywood_shadow_price_DA["datetime_ending_ept"] = (
    plywood_shadow_price_DA["file_date"]
    + pd.to_timedelta(
        plywood_shadow_price_DA["hour_ending"],
        unit="h"
    )
)

# rename to match your target format
plywood_shadow_price_DA = plywood_shadow_price_DA.rename(
    columns={
        "MONITORED_FACILITY_DESC": "nodename",
        "shadow_price": "shadow_price_DA"
    }
)

# reorder columns
plywood_shadow_price_DA = plywood_shadow_price_DA[
    ["datetime_ending_ept", "nodename", "shadow_price_DA"]
].sort_values(
    ["nodename", "datetime_ending_ept"]
).drop_duplicates(
    subset=["nodename", "datetime_ending_ept"],
    keep="last"
).reset_index(drop=True)

plywood_shadow_price_DA.head()

,datetime_ending_ept,nodename,shadow_price_DA
0,2026-04-14 18:00:00,PLYWOOD_115KV151A_1_LN,136.41
1,2026-04-14 19:00:00,PLYWOOD_115KV151A_1_LN,214.52
2,2026-04-14 20:00:00,PLYWOOD_115KV151A_1_LN,283.03
3,2026-04-14 21:00:00,PLYWOOD_115KV151A_1_LN,225.37
4,2026-04-14 22:00:00,PLYWOOD_115KV151A_1_LN,144.94


In [48]:
plywood_shadow_price['shadow_price'].max()

np.float64(2000.0)

## county

In [49]:
# Get all nodenames that belong to agg_nodename == "DOM" in hourly_load
dom_nodes = (
    hourly_load.loc[hourly_load["agg_nodename"] == "DOM", "nodename"]
    .dropna()
    .unique()
)

# Map those DOM nodenames to county_name using node_cty_map
dom_node_counties = (
    node_cty_map.loc[node_cty_map["nodename"].isin(dom_nodes), ["nodename", "county_name"]]
    .dropna(subset=["county_name"])
    .drop_duplicates()
)

# Get all county_name touched by those DOM nodenames
dom_counties = dom_node_counties["county_name"].unique()

# From those counties, get all nodenames in node_cty_map
all_nodes_in_dom_counties = (
    node_cty_map.loc[node_cty_map["county_name"].isin(dom_counties), "nodename"]
    .dropna()
    .unique()
)

# Select all rows from hourly_load whose nodename is in those nodenames
hourly_load_dom_counties = hourly_load.loc[
    hourly_load["nodename"].isin(all_nodes_in_dom_counties)
].copy()

In [50]:
hourly_load_county = hourly_load_dom_counties[["datetime_ending_ept", "agg_nodename", "nodename", "distribution_factor", "forecast_area", "forecast_load_mw", "distributed_load_mw"]].merge(node_cty_map[["nodename", "county_name", "latitude", "longitude", "state_short_name"]], on="nodename", how="left")

county_load = (
    hourly_load_county
    .groupby(
        ["datetime_ending_ept", "county_name", "state_short_name"],
        as_index=False
    )
    .agg(
        distributed_load_mw=("distributed_load_mw", "sum"),
        latitude=("latitude", "mean"),
        longitude=("longitude", "mean")
    )
    .sort_values(["datetime_ending_ept", "county_name"])
)

In [51]:
county_geo = (
    county_load
    .groupby("county_name", as_index=False)
    .agg(
        latitude=("latitude", "mean"),
        longitude=("longitude", "mean")
    )
)

# round to 2 decimals
county_geo["latitude"] = county_geo["latitude"].round(2)
county_geo["longitude"] = county_geo["longitude"].round(2)

county_load = county_load[["datetime_ending_ept", "county_name", "state_short_name", "distributed_load_mw"]].merge(county_geo, on="county_name", how="left").copy()

In [52]:
county_load = county_load[(county_load["datetime_ending_ept"] >= "2026-04-01")].copy()

In [53]:
county_load["datetime_ending_ept"] = pd.to_datetime(county_load["datetime_ending_ept"])

county_load_shadow_price = county_load.merge(
    plywood_shadow_price[["datetime_ending_ept", "shadow_price"]],
    on=["datetime_ending_ept"],
    how="left"
)

county_load_shadow_price["shadow_price"] = county_load_shadow_price["shadow_price"].fillna(0)

In [54]:
# row-level filter
df_filtered = county_load_shadow_price[
    (county_load_shadow_price["shadow_price"].notna()) &
    (county_load_shadow_price["shadow_price"] != 0)
].copy()

# keep only counties with mean load > 20
county_load_shadow = df_filtered[
    df_filtered.groupby("county_name")["distributed_load_mw"].transform("max") > 20
].copy()

In [55]:
df = county_load_shadow.copy()

feature_cols = ["distributed_load_mw"] 

# 3. correlation between each feature and shadow_price for each nodename
corr_by_node = (
    df.groupby("county_name")
      .apply(lambda g: g[feature_cols].corrwith(g["shadow_price"]))
      .reset_index()
)

corr_by_node.sort_values(by="distributed_load_mw", ascending=False)

,county_name,distributed_load_mw
37,VA_Falls Church,0.494817
25,VA_Campbell,0.494302
79,VA_Spotsylvania,0.493806
41,VA_Fredericksburg,0.492463
26,VA_Caroline,0.486491
...,...,...
58,VA_Mecklenburg,-0.169581
55,VA_Loudoun,-0.175681
4,NC_Bertie,-0.259377
12,NC_Northampton,-0.290034


In [56]:
corr_by_node[corr_by_node["county_name"] == "VA_Fairfax"]

,county_name,distributed_load_mw
36,VA_Fairfax,0.311733


### plot

In [57]:
county_load_shadow_price = county_load_shadow_price.merge(
    plywood_shadow_price_DA[["datetime_ending_ept", "shadow_price_DA"]],
    on=["datetime_ending_ept"],
    how="left"
)

county_load_shadow_price["shadow_price_DA"] = county_load_shadow_price["shadow_price_DA"].fillna(0)

In [58]:
df = county_load_shadow_price.copy()

node = "VA_Halifax"

sub = df[df["county_name"] == node].copy()

# ensure datetime is sorted
sub["datetime_ending_ept"] = pd.to_datetime(sub["datetime_ending_ept"])
sub = sub.sort_values("datetime_ending_ept")

fig = go.Figure()

# Left y-axis: distributed load
fig.add_trace(
    go.Scatter(
        x=sub["datetime_ending_ept"],
        y=sub["distributed_load_mw"],
        name="Distributed Load (MW)",
        line=dict(color="blue"),
        yaxis="y1"
    )
)

# Right y-axis: shadow price
fig.add_trace(
    go.Scatter(
        x=sub["datetime_ending_ept"],
        y=sub["shadow_price"],
        name="Shadow Price",
        line=dict(color="red"),
        yaxis="y2"
    )
)

fig.add_trace(
    go.Scatter(
        x=sub["datetime_ending_ept"],
        y=sub["shadow_price_DA"],
        name="Shadow Price DA",
        line=dict(color="green"),
        yaxis="y2"
    )
)

# Layout with dual axis
fig.update_layout(
    title=f"{node}: Load vs Shadow Price",
    xaxis=dict(title="Datetime"),
    
    yaxis=dict(
        title="Distributed Load (MW)",
        side="left"
    ),
    
    yaxis2=dict(
        title="Shadow Price",
        overlaying="y",
        side="right"
    ),
    
    legend=dict(x=0.01, y=0.99),
    height=500,
    width=2000
)

fig.show()

# LADYSMTH

## shadow

In [59]:
# copy
df = RT_constr[RT_constr["MONITORED_FACILITY_DESC"].str.contains("LADYSMTH")].copy()

# hourly columns
hour_cols = [str(i) for i in range(1, 25)]

# keep only rows that have at least one hourly value
df = df.dropna(subset=hour_cols, how="all")

df["file_date"] = pd.to_datetime(df["file_date"])

# reshape to long format
ladysmth_shadow_price = df.melt(
    id_vars=["MONITORED_FACILITY_DESC", "file_date"],
    value_vars=hour_cols,
    var_name="hour_ending",
    value_name="shadow_price"
)

# drop null hourly values
ladysmth_shadow_price = ladysmth_shadow_price.dropna(subset=["shadow_price"]).copy()

# convert HE to int
ladysmth_shadow_price["hour_ending"] = ladysmth_shadow_price["hour_ending"].astype(int)

# build datetime_ending_ept
ladysmth_shadow_price["datetime_ending_ept"] = (
    ladysmth_shadow_price["file_date"] + pd.to_timedelta(
        ladysmth_shadow_price["hour_ending"], unit="h"
    )
)

# rename to match your target format
ladysmth_shadow_price = ladysmth_shadow_price.rename(
    columns={"MONITORED_FACILITY_DESC": "nodename"}
)

# reorder columns
ladysmth_shadow_price = ladysmth_shadow_price[
    ["datetime_ending_ept", "nodename", "shadow_price"]
].sort_values(
    ["nodename", "datetime_ending_ept"]
).drop_duplicates(
    subset=["nodename", "datetime_ending_ept"],
    keep="last"
).reset_index(drop=True)

ladysmth_shadow_price.head()

,datetime_ending_ept,nodename,shadow_price
0,2026-04-08 10:00:00,LADYSMTH H1T575 CB 500 KV,4.50
1,2026-04-08 13:00:00,LADYSMTH H1T575 CB 500 KV,3.09
2,2026-04-08 14:00:00,LADYSMTH H1T575 CB 500 KV,46.84
3,2026-04-08 15:00:00,LADYSMTH H1T575 CB 500 KV,46.97
4,2026-04-08 16:00:00,LADYSMTH H1T575 CB 500 KV,44.14


In [60]:
# copy
df = DA_constr[DA_constr["MONITORED_FACILITY_DESC"].str.contains("LADYSMTH")].copy()

# hourly columns
hour_cols = [str(i) for i in range(1, 25)]

# keep only rows that have at least one hourly value
df = df.dropna(subset=hour_cols, how="all")

df["file_date"] = pd.to_datetime(df["file_date"])

# reshape to long format
ladysmth_shadow_price_DA = df.melt(
    id_vars=["MONITORED_FACILITY_DESC", "file_date"],
    value_vars=hour_cols,
    var_name="hour_ending",
    value_name="shadow_price"
)

# drop null hourly values
ladysmth_shadow_price_DA = ladysmth_shadow_price_DA.dropna(subset=["shadow_price"]).copy()

# convert HE to int
ladysmth_shadow_price_DA["hour_ending"] = ladysmth_shadow_price_DA["hour_ending"].astype(int)

# build datetime_ending_ept
ladysmth_shadow_price_DA["datetime_ending_ept"] = (
    ladysmth_shadow_price_DA["file_date"] + 
    pd.to_timedelta(ladysmth_shadow_price_DA["hour_ending"], unit="h")
)

# rename to match your target format
ladysmth_shadow_price_DA = ladysmth_shadow_price_DA.rename(
    columns={
        "MONITORED_FACILITY_DESC": "nodename",
        "shadow_price": "shadow_price_DA"
    }
)

# reorder columns
ladysmth_shadow_price_DA = ladysmth_shadow_price_DA[
    ["datetime_ending_ept", "nodename", "shadow_price_DA"]
].sort_values(
    ["nodename", "datetime_ending_ept"]
).drop_duplicates(
    subset=["nodename", "datetime_ending_ept"],
    keep="last"
).reset_index(drop=True)

ladysmth_shadow_price_DA.head()

,datetime_ending_ept,nodename,shadow_price_DA
0,2026-05-06 10:00:00,LADYSMTH HIT575 CB 500KV,40.08
1,2026-05-06 11:00:00,LADYSMTH HIT575 CB 500KV,28.71
2,2026-05-06 12:00:00,LADYSMTH HIT575 CB 500KV,50.89
3,2026-05-06 13:00:00,LADYSMTH HIT575 CB 500KV,63.74
4,2026-05-06 14:00:00,LADYSMTH HIT575 CB 500KV,76.71


In [61]:
ladysmth_shadow_price["shadow_price"].max()

np.float64(938.45)

## county

In [62]:
# Get all nodenames that belong to agg_nodename == "DOM" in hourly_load
dom_nodes = (
    hourly_load.loc[hourly_load["agg_nodename"] == "DOM", "nodename"]
    .dropna()
    .unique()
)

# Map those DOM nodenames to county_name using node_cty_map
dom_node_counties = (
    node_cty_map.loc[node_cty_map["nodename"].isin(dom_nodes), ["nodename", "county_name"]]
    .dropna(subset=["county_name"])
    .drop_duplicates()
)

# Get all county_name touched by those DOM nodenames
dom_counties = dom_node_counties["county_name"].unique()

# From those counties, get all nodenames in node_cty_map
all_nodes_in_dom_counties = (
    node_cty_map.loc[node_cty_map["county_name"].isin(dom_counties), "nodename"]
    .dropna()
    .unique()
)

# Select all rows from hourly_load whose nodename is in those nodenames
hourly_load_dom_counties = hourly_load.loc[
    hourly_load["nodename"].isin(all_nodes_in_dom_counties)
].copy()

In [63]:
hourly_load_county = hourly_load_dom_counties[["datetime_ending_ept", "agg_nodename", "nodename", "distribution_factor", "forecast_area", "forecast_load_mw", "distributed_load_mw"]].merge(node_cty_map[["nodename", "county_name", "latitude", "longitude", "state_short_name"]], on="nodename", how="left")

county_load = (
    hourly_load_county
    .groupby(
        ["datetime_ending_ept", "county_name", "state_short_name"],
        as_index=False
    )
    .agg(
        distributed_load_mw=("distributed_load_mw", "sum"),
        latitude=("latitude", "mean"),
        longitude=("longitude", "mean")
    )
    .sort_values(["datetime_ending_ept", "county_name"])
)

In [64]:
county_geo = (
    county_load
    .groupby("county_name", as_index=False)
    .agg(
        latitude=("latitude", "mean"),
        longitude=("longitude", "mean")
    )
)

# round to 2 decimals
county_geo["latitude"] = county_geo["latitude"].round(2)
county_geo["longitude"] = county_geo["longitude"].round(2)

county_load = county_load[["datetime_ending_ept", "county_name", "state_short_name", "distributed_load_mw"]].merge(county_geo, on="county_name", how="left").copy()

In [65]:
county_load = county_load[(county_load["datetime_ending_ept"] >= "2026-05-01")].copy()

In [66]:
county_load["datetime_ending_ept"] = pd.to_datetime(county_load["datetime_ending_ept"])

county_load_shadow_price = county_load.merge(
    ladysmth_shadow_price[["datetime_ending_ept", "shadow_price"]],
    on=["datetime_ending_ept"],
    how="left"
)

county_load_shadow_price["shadow_price"] = county_load_shadow_price["shadow_price"].fillna(0)

In [67]:
# row-level filter
df_filtered = county_load_shadow_price[
    (county_load_shadow_price["shadow_price"].notna()) &
    (county_load_shadow_price["shadow_price"] != 0)
].copy()

# keep only counties with mean load > 20
county_load_shadow = df_filtered[
    df_filtered.groupby("county_name")["distributed_load_mw"].transform("max") > 20
].copy()

In [68]:
df = county_load_shadow.copy()

feature_cols = ["distributed_load_mw"] 

# 3. correlation between each feature and shadow_price for each nodename
corr_by_node = (
    df.groupby("county_name")
      .apply(lambda g: g[feature_cols].corrwith(g["shadow_price"]))
      .reset_index()
)

corr_by_node.sort_values(by="distributed_load_mw", ascending=False)

,county_name,distributed_load_mw
16,VA_Alleghany,0.567326
46,VA_Harrisonburg,0.564612
70,VA_Prince George,0.559179
1,MD_Charles,0.558747
2,MD_Montgomery,0.554212
...,...,...
33,VA_Essex,-0.076671
9,NC_Hertford,-0.087753
8,NC_Halifax,-0.101531
4,NC_Bertie,-0.105212


In [69]:
corr_by_node[corr_by_node["county_name"] == "VA_Caroline"]

,county_name,distributed_load_mw
25,VA_Caroline,0.462898


### plot

In [70]:
county_load_shadow_price = county_load_shadow_price.merge(
    ladysmth_shadow_price_DA[["datetime_ending_ept", "shadow_price_DA"]],
    on=["datetime_ending_ept"],
    how="left"
)

county_load_shadow_price["shadow_price_DA"] = county_load_shadow_price["shadow_price_DA"].fillna(0)

In [71]:
df = county_load_shadow_price.copy()

node = "VA_Caroline"

sub = df[df["county_name"] == node].copy()

# ensure datetime is sorted
sub["datetime_ending_ept"] = pd.to_datetime(sub["datetime_ending_ept"])
sub = sub.sort_values("datetime_ending_ept")

fig = go.Figure()

# Left y-axis: distributed load
fig.add_trace(
    go.Scatter(
        x=sub["datetime_ending_ept"],
        y=sub["distributed_load_mw"],
        name="Distributed Load (MW)",
        line=dict(color="blue"),
        yaxis="y1"
    )
)

# Right y-axis: shadow price
fig.add_trace(
    go.Scatter(
        x=sub["datetime_ending_ept"],
        y=sub["shadow_price"],
        name="Shadow Price",
        line=dict(color="red"),
        yaxis="y2"
    )
)

fig.add_trace(
    go.Scatter(
        x=sub["datetime_ending_ept"],
        y=sub["shadow_price_DA"],
        name="Shadow Price DA",
        line=dict(color="green"),
        yaxis="y2"
    )
)


# Layout with dual axis
fig.update_layout(
    title=f"{node}: Load vs Shadow Price",
    xaxis=dict(title="Datetime"),
    
    yaxis=dict(
        title="Distributed Load (MW)",
        side="left"
    ),
    
    yaxis2=dict(
        title="Shadow Price",
        overlaying="y",
        side="right"
    ),
    
    legend=dict(x=0.01, y=0.99),
    height=500,
    width=2000
)

fig.show()

# LAKRIDGE

## shadow

In [93]:
# copy
df = RT_constr[RT_constr["MONITORED_FACILITY_DESC"].str.contains("LAKRIDGE")].copy()

# hourly columns
hour_cols = [str(i) for i in range(1, 25)]

# keep only rows that have at least one hourly value
df = df.dropna(subset=hour_cols, how="all")

df["file_date"] = pd.to_datetime(df["file_date"])

# reshape to long format
lakridge_shadow_price = df.melt(
    id_vars=["MONITORED_FACILITY_DESC", "file_date"],
    value_vars=hour_cols,
    var_name="hour_ending",
    value_name="shadow_price"
)

# drop null hourly values
lakridge_shadow_price = lakridge_shadow_price.dropna(subset=["shadow_price"]).copy()

# convert HE to int
lakridge_shadow_price["hour_ending"] = lakridge_shadow_price["hour_ending"].astype(int)

# build datetime_ending_ept
lakridge_shadow_price["datetime_ending_ept"] = (
    lakridge_shadow_price["file_date"] + pd.to_timedelta(
        lakridge_shadow_price["hour_ending"], unit="h"
    )
)

# rename to match your target format
lakridge_shadow_price = lakridge_shadow_price.rename(
    columns={"MONITORED_FACILITY_DESC": "nodename"}
)

# reorder columns
lakridge_shadow_price = lakridge_shadow_price[
    ["datetime_ending_ept", "nodename", "shadow_price"]
].sort_values(
    ["nodename", "datetime_ending_ept"]
).drop_duplicates(
    subset=["nodename", "datetime_ending_ept"],
    keep="last"
).reset_index(drop=True)

lakridge_shadow_price.head()

,datetime_ending_ept,nodename,shadow_price
0,2026-05-18 18:00:00,DUMFRIES-LAKRIDGE 2022B B 230 KV,246.92
1,2026-04-20 09:00:00,LAKRIDGE-RAVNSWTH 2022C B 230 KV,41.65
2,2026-04-20 10:00:00,LAKRIDGE-RAVNSWTH 2022C B 230 KV,112.82
3,2026-04-20 11:00:00,LAKRIDGE-RAVNSWTH 2022C B 230 KV,19.14
4,2026-04-20 12:00:00,LAKRIDGE-RAVNSWTH 2022C B 230 KV,387.94


In [94]:
# copy
df = DA_constr[DA_constr["MONITORED_FACILITY_DESC"].str.contains("LAKRIDGE")].copy()

# hourly columns
hour_cols = [str(i) for i in range(1, 25)]

# keep only rows that have at least one hourly value
df = df.dropna(subset=hour_cols, how="all")

df["file_date"] = pd.to_datetime(df["file_date"])

# reshape to long format
lakridge_shadow_price_DA = df.melt(
    id_vars=["MONITORED_FACILITY_DESC", "file_date"],
    value_vars=hour_cols,
    var_name="hour_ending",
    value_name="shadow_price"
)

# drop null hourly values
lakridge_shadow_price_DA = lakridge_shadow_price_DA.dropna(subset=["shadow_price"]).copy()

# convert HE to int
lakridge_shadow_price_DA["hour_ending"] = lakridge_shadow_price_DA["hour_ending"].astype(int)

# build datetime_ending_ept
lakridge_shadow_price_DA["datetime_ending_ept"] = (
    lakridge_shadow_price_DA["file_date"] + 
    pd.to_timedelta(lakridge_shadow_price_DA["hour_ending"], unit="h")
)

# rename to match your target format
lakridge_shadow_price_DA = lakridge_shadow_price_DA.rename(
    columns={
        "MONITORED_FACILITY_DESC": "nodename",
        "shadow_price": "shadow_price_DA"
    }
)

# reorder columns
lakridge_shadow_price_DA = lakridge_shadow_price_DA[
    ["datetime_ending_ept", "nodename", "shadow_price_DA"]
].sort_values(
    ["nodename", "datetime_ending_ept"]
).drop_duplicates(
    subset=["nodename", "datetime_ending_ept"],
    keep="last"
).reset_index(drop=True)

lakridge_shadow_price_DA.head()

,datetime_ending_ept,nodename,shadow_price_DA


In [95]:
lakridge_shadow_price["shadow_price"].max()

np.float64(387.94)

## county

In [96]:
# Get all nodenames that belong to agg_nodename == "DOM" in hourly_load
dom_nodes = (
    hourly_load.loc[hourly_load["agg_nodename"] == "DOM", "nodename"]
    .dropna()
    .unique()
)

# Map those DOM nodenames to county_name using node_cty_map
dom_node_counties = (
    node_cty_map.loc[node_cty_map["nodename"].isin(dom_nodes), ["nodename", "county_name"]]
    .dropna(subset=["county_name"])
    .drop_duplicates()
)

# Get all county_name touched by those DOM nodenames
dom_counties = dom_node_counties["county_name"].unique()

# From those counties, get all nodenames in node_cty_map
all_nodes_in_dom_counties = (
    node_cty_map.loc[node_cty_map["county_name"].isin(dom_counties), "nodename"]
    .dropna()
    .unique()
)

# Select all rows from hourly_load whose nodename is in those nodenames
hourly_load_dom_counties = hourly_load.loc[
    hourly_load["nodename"].isin(all_nodes_in_dom_counties)
].copy()

In [97]:
hourly_load_county = hourly_load_dom_counties[["datetime_ending_ept", "agg_nodename", "nodename", "distribution_factor", "forecast_area", "forecast_load_mw", "distributed_load_mw"]].merge(node_cty_map[["nodename", "county_name", "latitude", "longitude", "state_short_name"]], on="nodename", how="left")

county_load = (
    hourly_load_county
    .groupby(
        ["datetime_ending_ept", "county_name", "state_short_name"],
        as_index=False
    )
    .agg(
        distributed_load_mw=("distributed_load_mw", "sum"),
        latitude=("latitude", "mean"),
        longitude=("longitude", "mean")
    )
    .sort_values(["datetime_ending_ept", "county_name"])
)

In [98]:
county_geo = (
    county_load
    .groupby("county_name", as_index=False)
    .agg(
        latitude=("latitude", "mean"),
        longitude=("longitude", "mean")
    )
)

# round to 2 decimals
county_geo["latitude"] = county_geo["latitude"].round(2)
county_geo["longitude"] = county_geo["longitude"].round(2)

county_load = county_load[["datetime_ending_ept", "county_name", "state_short_name", "distributed_load_mw"]].merge(county_geo, on="county_name", how="left").copy()

In [99]:
county_load = county_load[(county_load["datetime_ending_ept"] >= "2026-04-20")].copy()

In [100]:
county_load["datetime_ending_ept"] = pd.to_datetime(county_load["datetime_ending_ept"])

county_load_shadow_price = county_load.merge(
    lakridge_shadow_price[["datetime_ending_ept", "shadow_price"]],
    on=["datetime_ending_ept"],
    how="left"
)

county_load_shadow_price["shadow_price"] = county_load_shadow_price["shadow_price"].fillna(0)

In [101]:
# row-level filter
df_filtered = county_load_shadow_price[
    (county_load_shadow_price["shadow_price"].notna()) &
    (county_load_shadow_price["shadow_price"] != 0)
].copy()

# keep only counties with mean load > 20
county_load_shadow = df_filtered[
    df_filtered.groupby("county_name")["distributed_load_mw"].transform("max") > 20
].copy()

In [102]:
df = county_load_shadow.copy()

feature_cols = ["distributed_load_mw"] 

# 3. correlation between each feature and shadow_price for each nodename
corr_by_node = (
    df.groupby("county_name")
      .apply(lambda g: g[feature_cols].corrwith(g["shadow_price"]))
      .reset_index()
)

corr_by_node.sort_values(by="distributed_load_mw", ascending=False)

,county_name,distributed_load_mw
83,VA_Westmoreland,0.489027
39,VA_Goochland,0.486047
16,VA_Alleghany,0.435754
58,VA_Newport News,0.433524
51,VA_Lancaster,0.431240
...,...,...
79,VA_Surry,0.150783
7,NC_Gates,0.120812
75,VA_Southampton,0.099585
50,VA_King William,0.093113


In [103]:
corr_by_node[corr_by_node["county_name"] == "VA_Prince William"]

,county_name,distributed_load_mw
69,VA_Prince William,0.238699


### plot

In [104]:
county_load_shadow_price = county_load_shadow_price.merge(
    lakridge_shadow_price_DA[["datetime_ending_ept", "shadow_price_DA"]],
    on=["datetime_ending_ept"],
    how="left"
)

county_load_shadow_price["shadow_price_DA"] = county_load_shadow_price["shadow_price_DA"].fillna(0)

In [105]:
df = county_load_shadow_price.copy()

node = "VA_Prince William"

sub = df[df["county_name"] == node].copy()

# ensure datetime is sorted
sub["datetime_ending_ept"] = pd.to_datetime(sub["datetime_ending_ept"])
sub = sub.sort_values("datetime_ending_ept")

fig = go.Figure()

# Left y-axis: distributed load
fig.add_trace(
    go.Scatter(
        x=sub["datetime_ending_ept"],
        y=sub["distributed_load_mw"],
        name="Distributed Load (MW)",
        line=dict(color="blue"),
        yaxis="y1"
    )
)

# Right y-axis: shadow price
fig.add_trace(
    go.Scatter(
        x=sub["datetime_ending_ept"],
        y=sub["shadow_price"],
        name="Shadow Price",
        line=dict(color="red"),
        yaxis="y2"
    )
)

fig.add_trace(
    go.Scatter(
        x=sub["datetime_ending_ept"],
        y=sub["shadow_price_DA"],
        name="Shadow Price DA",
        line=dict(color="green"),
        yaxis="y2"
    )
)


# Layout with dual axis
fig.update_layout(
    title=f"{node}: Load vs Shadow Price",
    xaxis=dict(title="Datetime"),
    
    yaxis=dict(
        title="Distributed Load (MW)",
        side="left"
    ),
    
    yaxis2=dict(
        title="Shadow Price",
        overlaying="y",
        side="right"
    ),
    
    legend=dict(x=0.01, y=0.99),
    height=500,
    width=2000
)

fig.show()

# OX4

## shadow

In [110]:
# copy
df = RT_constr[RT_constr["MONITORED_FACILITY_DESC"].str.contains("OX4")].copy()

# hourly columns
hour_cols = [str(i) for i in range(1, 25)]

# keep only rows that have at least one hourly value
df = df.dropna(subset=hour_cols, how="all")

df["file_date"] = pd.to_datetime(df["file_date"])

# reshape to long format
ox_shadow_price = df.melt(
    id_vars=["MONITORED_FACILITY_DESC", "file_date"],
    value_vars=hour_cols,
    var_name="hour_ending",
    value_name="shadow_price"
)

# drop null hourly values
ox_shadow_price = ox_shadow_price.dropna(subset=["shadow_price"]).copy()

# convert HE to int
ox_shadow_price["hour_ending"] = ox_shadow_price["hour_ending"].astype(int)

# build datetime_ending_ept
ox_shadow_price["datetime_ending_ept"] = (
    ox_shadow_price["file_date"] + pd.to_timedelta(
        ox_shadow_price["hour_ending"], unit="h"
    )
)

# rename to match your target format
ox_shadow_price = ox_shadow_price.rename(
    columns={"MONITORED_FACILITY_DESC": "nodename"}
)

# reorder columns
ox_shadow_price = ox_shadow_price[
    ["datetime_ending_ept", "nodename", "shadow_price"]
].sort_values(
    ["nodename", "datetime_ending_ept"]
).drop_duplicates(
    subset=["nodename", "datetime_ending_ept"],
    keep="last"
).reset_index(drop=True)

ox_shadow_price.head()

,datetime_ending_ept,nodename,shadow_price
0,2026-04-15 16:00:00,OX4 TX2 XFORMER H 500 KV,1000.0
1,2026-04-15 17:00:00,OX4 TX2 XFORMER H 500 KV,2000.0
2,2026-04-15 18:00:00,OX4 TX2 XFORMER H 500 KV,2000.0
3,2026-04-15 19:00:00,OX4 TX2 XFORMER H 500 KV,2000.0
4,2026-04-15 20:00:00,OX4 TX2 XFORMER H 500 KV,500.0


In [111]:
# copy
df = DA_constr[DA_constr["MONITORED_FACILITY_DESC"].str.contains("OX4")].copy()

# hourly columns
hour_cols = [str(i) for i in range(1, 25)]

# keep only rows that have at least one hourly value
df = df.dropna(subset=hour_cols, how="all")

df["file_date"] = pd.to_datetime(df["file_date"])

# reshape to long format
ox_shadow_price_DA = df.melt(
    id_vars=["MONITORED_FACILITY_DESC", "file_date"],
    value_vars=hour_cols,
    var_name="hour_ending",
    value_name="shadow_price"
)

# drop null hourly values
ox_shadow_price_DA = ox_shadow_price_DA.dropna(subset=["shadow_price"]).copy()

# convert HE to int
ox_shadow_price_DA["hour_ending"] = ox_shadow_price_DA["hour_ending"].astype(int)

# build datetime_ending_ept
ox_shadow_price_DA["datetime_ending_ept"] = (
    ox_shadow_price_DA["file_date"] + 
    pd.to_timedelta(ox_shadow_price_DA["hour_ending"], unit="h")
)

# rename to match your target format
ox_shadow_price_DA = ox_shadow_price_DA.rename(
    columns={
        "MONITORED_FACILITY_DESC": "nodename",
        "shadow_price": "shadow_price_DA"
    }
)

# reorder columns
ox_shadow_price_DA = ox_shadow_price_DA[
    ["datetime_ending_ept", "nodename", "shadow_price_DA"]
].sort_values(
    ["nodename", "datetime_ending_ept"]
).drop_duplicates(
    subset=["nodename", "datetime_ending_ept"],
    keep="last"
).reset_index(drop=True)

ox_shadow_price_DA.head()

,datetime_ending_ept,nodename,shadow_price_DA
0,2026-04-23 17:00:00,OX4_T1_500TX2_XF,1.06
1,2026-04-23 18:00:00,OX4_T1_500TX2_XF,138.64
2,2026-05-05 13:00:00,OX4_T1_500TX2_XF,68.06
3,2026-05-05 14:00:00,OX4_T1_500TX2_XF,98.79
4,2026-05-05 15:00:00,OX4_T1_500TX2_XF,105.53


In [112]:
ox_shadow_price["shadow_price"].max()

np.float64(2000.0)

## county

In [113]:
# Get all nodenames that belong to agg_nodename == "DOM" in hourly_load
dom_nodes = (
    hourly_load.loc[hourly_load["agg_nodename"] == "DOM", "nodename"]
    .dropna()
    .unique()
)

# Map those DOM nodenames to county_name using node_cty_map
dom_node_counties = (
    node_cty_map.loc[node_cty_map["nodename"].isin(dom_nodes), ["nodename", "county_name"]]
    .dropna(subset=["county_name"])
    .drop_duplicates()
)

# Get all county_name touched by those DOM nodenames
dom_counties = dom_node_counties["county_name"].unique()

# From those counties, get all nodenames in node_cty_map
all_nodes_in_dom_counties = (
    node_cty_map.loc[node_cty_map["county_name"].isin(dom_counties), "nodename"]
    .dropna()
    .unique()
)

# Select all rows from hourly_load whose nodename is in those nodenames
hourly_load_dom_counties = hourly_load.loc[
    hourly_load["nodename"].isin(all_nodes_in_dom_counties)
].copy()

In [114]:
hourly_load_county = hourly_load_dom_counties[["datetime_ending_ept", "agg_nodename", "nodename", "distribution_factor", "forecast_area", "forecast_load_mw", "distributed_load_mw"]].merge(node_cty_map[["nodename", "county_name", "latitude", "longitude", "state_short_name"]], on="nodename", how="left")

county_load = (
    hourly_load_county
    .groupby(
        ["datetime_ending_ept", "county_name", "state_short_name"],
        as_index=False
    )
    .agg(
        distributed_load_mw=("distributed_load_mw", "sum"),
        latitude=("latitude", "mean"),
        longitude=("longitude", "mean")
    )
    .sort_values(["datetime_ending_ept", "county_name"])
)

In [115]:
county_geo = (
    county_load
    .groupby("county_name", as_index=False)
    .agg(
        latitude=("latitude", "mean"),
        longitude=("longitude", "mean")
    )
)

# round to 2 decimals
county_geo["latitude"] = county_geo["latitude"].round(2)
county_geo["longitude"] = county_geo["longitude"].round(2)

county_load = county_load[["datetime_ending_ept", "county_name", "state_short_name", "distributed_load_mw"]].merge(county_geo, on="county_name", how="left").copy()

In [116]:
county_load = county_load[(county_load["datetime_ending_ept"] >= "2026-04-15")].copy()

In [117]:
county_load["datetime_ending_ept"] = pd.to_datetime(county_load["datetime_ending_ept"])

county_load_shadow_price = county_load.merge(
    ox_shadow_price[["datetime_ending_ept", "shadow_price"]],
    on=["datetime_ending_ept"],
    how="left"
)

county_load_shadow_price["shadow_price"] = county_load_shadow_price["shadow_price"].fillna(0)

In [118]:
# row-level filter
df_filtered = county_load_shadow_price[
    (county_load_shadow_price["shadow_price"].notna()) &
    (county_load_shadow_price["shadow_price"] != 0)
].copy()

# keep only counties with mean load > 20
county_load_shadow = df_filtered[
    df_filtered.groupby("county_name")["distributed_load_mw"].transform("max") > 20
].copy()

In [119]:
df = county_load_shadow.copy()

feature_cols = ["distributed_load_mw"] 

# 3. correlation between each feature and shadow_price for each nodename
corr_by_node = (
    df.groupby("county_name")
      .apply(lambda g: g[feature_cols].corrwith(g["shadow_price"]))
      .reset_index()
)

corr_by_node.sort_values(by="distributed_load_mw", ascending=False)

,county_name,distributed_load_mw
44,VA_Greensville,0.219103
15,VA_Albemarle,0.194404
7,NC_Gates,0.162426
64,VA_Nottoway,0.120498
57,VA_Manassas,0.120400
...,...,...
12,NC_Northampton,-0.178023
4,NC_Bertie,-0.180873
14,NC_Perquimans,-0.183565
78,VA_Shenandoah,-0.196807


In [120]:
corr_by_node[corr_by_node["county_name"] == "VA_Fairfax"]

,county_name,distributed_load_mw
36,VA_Fairfax,0.080207


### plot

In [121]:
county_load_shadow_price = county_load_shadow_price.merge(
    ox_shadow_price_DA[["datetime_ending_ept", "shadow_price_DA"]],
    on=["datetime_ending_ept"],
    how="left"
)

county_load_shadow_price["shadow_price_DA"] = county_load_shadow_price["shadow_price_DA"].fillna(0)

In [122]:
df = county_load_shadow_price.copy()

node = "VA_Fairfax"

sub = df[df["county_name"] == node].copy()

# ensure datetime is sorted
sub["datetime_ending_ept"] = pd.to_datetime(sub["datetime_ending_ept"])
sub = sub.sort_values("datetime_ending_ept")

fig = go.Figure()

# Left y-axis: distributed load
fig.add_trace(
    go.Scatter(
        x=sub["datetime_ending_ept"],
        y=sub["distributed_load_mw"],
        name="Distributed Load (MW)",
        line=dict(color="blue"),
        yaxis="y1"
    )
)

# Right y-axis: shadow price
fig.add_trace(
    go.Scatter(
        x=sub["datetime_ending_ept"],
        y=sub["shadow_price"],
        name="Shadow Price",
        line=dict(color="red"),
        yaxis="y2"
    )
)

fig.add_trace(
    go.Scatter(
        x=sub["datetime_ending_ept"],
        y=sub["shadow_price_DA"],
        name="Shadow Price DA",
        line=dict(color="green"),
        yaxis="y2"
    )
)


# Layout with dual axis
fig.update_layout(
    title=f"{node}: Load vs Shadow Price",
    xaxis=dict(title="Datetime"),
    
    yaxis=dict(
        title="Distributed Load (MW)",
        side="left"
    ),
    
    yaxis2=dict(
        title="Shadow Price",
        overlaying="y",
        side="right"
    ),
    
    legend=dict(x=0.01, y=0.99),
    height=500,
    width=2000
)

fig.show()

# AQUAHAR

## shadow

In [97]:
# copy
df = RT_constr[RT_constr["MONITORED_FACILITY_DESC"].str.contains("AQUAHAR")].copy()

# hourly columns
hour_cols = [str(i) for i in range(1, 25)]

# keep only rows that have at least one hourly value
df = df.dropna(subset=hour_cols, how="all")

df["file_date"] = pd.to_datetime(df["file_date"])

# reshape to long format
aquahar_shadow_price = df.melt(
    id_vars=["MONITORED_FACILITY_DESC", "file_date"],
    value_vars=hour_cols,
    var_name="hour_ending",
    value_name="shadow_price"
)

# drop null hourly values
aquahar_shadow_price = aquahar_shadow_price.dropna(subset=["shadow_price"]).copy()

# convert HE to int
aquahar_shadow_price["hour_ending"] = aquahar_shadow_price["hour_ending"].astype(int)

# build datetime_ending_ept
aquahar_shadow_price["datetime_ending_ept"] = (
    aquahar_shadow_price["file_date"] + pd.to_timedelta(
        aquahar_shadow_price["hour_ending"], unit="h"
    )
)

# rename to match your target format
aquahar_shadow_price = aquahar_shadow_price.rename(
    columns={"MONITORED_FACILITY_DESC": "nodename"}
)

# reorder columns
aquahar_shadow_price = aquahar_shadow_price[
    ["datetime_ending_ept", "nodename", "shadow_price"]
].sort_values(
    ["nodename", "datetime_ending_ept"]
).drop_duplicates(
    subset=["nodename", "datetime_ending_ept"],
    keep="last"
).reset_index(drop=True)

aquahar_shadow_price.head()

,datetime_ending_ept,nodename,shadow_price
0,2026-05-18 15:00:00,AQUAHAR-CRANESCR 2104T A 230 KV,443.39
1,2026-05-18 16:00:00,AQUAHAR-CRANESCR 2104T A 230 KV,1506.80
2,2026-05-18 17:00:00,AQUAHAR-CRANESCR 2104T A 230 KV,670.12
3,2026-05-18 18:00:00,AQUAHAR-CRANESCR 2104T A 230 KV,1079.32
4,2026-05-18 19:00:00,AQUAHAR-CRANESCR 2104T A 230 KV,538.98


In [98]:
# copy
df = DA_constr[DA_constr["MONITORED_FACILITY_DESC"].str.contains("AQUAHAR")].copy()

# hourly columns
hour_cols = [str(i) for i in range(1, 25)]

# keep only rows that have at least one hourly value
df = df.dropna(subset=hour_cols, how="all")

df["file_date"] = pd.to_datetime(df["file_date"])

# reshape to long format
aquahar_shadow_price_DA = df.melt(
    id_vars=["MONITORED_FACILITY_DESC", "file_date"],
    value_vars=hour_cols,
    var_name="hour_ending",
    value_name="shadow_price"
)

# drop null hourly values
aquahar_shadow_price_DA = aquahar_shadow_price_DA.dropna(subset=["shadow_price"]).copy()

# convert HE to int
aquahar_shadow_price_DA["hour_ending"] = aquahar_shadow_price_DA["hour_ending"].astype(int)

# build datetime_ending_ept
aquahar_shadow_price_DA["datetime_ending_ept"] = (
    aquahar_shadow_price_DA["file_date"] + 
    pd.to_timedelta(aquahar_shadow_price_DA["hour_ending"], unit="h")
)

# rename to match your target format
aquahar_shadow_price_DA = aquahar_shadow_price_DA.rename(
    columns={
        "MONITORED_FACILITY_DESC": "nodename",
        "shadow_price": "shadow_price_DA"
    }
)

# reorder columns
aquahar_shadow_price_DA = aquahar_shadow_price_DA[
    ["datetime_ending_ept", "nodename", "shadow_price_DA"]
].sort_values(
    ["nodename", "datetime_ending_ept"]
).drop_duplicates(
    subset=["nodename", "datetime_ending_ept"],
    keep="last"
).reset_index(drop=True)

aquahar_shadow_price_DA.head()

,datetime_ending_ept,nodename,shadow_price_DA
0,2026-05-20 08:00:00,AQUAHAR_230KVAQU-CRAN_1_LN,8.03
1,2026-05-20 09:00:00,AQUAHAR_230KVAQU-CRAN_1_LN,101.08
2,2026-05-20 10:00:00,AQUAHAR_230KVAQU-CRAN_1_LN,272.01
3,2026-05-20 11:00:00,AQUAHAR_230KVAQU-CRAN_1_LN,283.15
4,2026-05-20 12:00:00,AQUAHAR_230KVAQU-CRAN_1_LN,368.94


In [99]:
aquahar_shadow_price["shadow_price"].max()

np.float64(2000.0)

## county

In [100]:
# Get all nodenames that belong to agg_nodename == "DOM" in hourly_load
dom_nodes = (
    hourly_load.loc[hourly_load["agg_nodename"] == "DOM", "nodename"]
    .dropna()
    .unique()
)

# Map those DOM nodenames to county_name using node_cty_map
dom_node_counties = (
    node_cty_map.loc[node_cty_map["nodename"].isin(dom_nodes), ["nodename", "county_name"]]
    .dropna(subset=["county_name"])
    .drop_duplicates()
)

# Get all county_name touched by those DOM nodenames
dom_counties = dom_node_counties["county_name"].unique()

# From those counties, get all nodenames in node_cty_map
all_nodes_in_dom_counties = (
    node_cty_map.loc[node_cty_map["county_name"].isin(dom_counties), "nodename"]
    .dropna()
    .unique()
)

# Select all rows from hourly_load whose nodename is in those nodenames
hourly_load_dom_counties = hourly_load.loc[
    hourly_load["nodename"].isin(all_nodes_in_dom_counties)
].copy()

In [101]:
hourly_load_county = hourly_load_dom_counties[["datetime_ending_ept", "agg_nodename", "nodename", "distribution_factor", "forecast_area", "forecast_load_mw", "distributed_load_mw"]].merge(node_cty_map[["nodename", "county_name", "latitude", "longitude", "state_short_name"]], on="nodename", how="left")

county_load = (
    hourly_load_county
    .groupby(
        ["datetime_ending_ept", "county_name", "state_short_name"],
        as_index=False
    )
    .agg(
        distributed_load_mw=("distributed_load_mw", "sum"),
        latitude=("latitude", "mean"),
        longitude=("longitude", "mean")
    )
    .sort_values(["datetime_ending_ept", "county_name"])
)

In [102]:
county_geo = (
    county_load
    .groupby("county_name", as_index=False)
    .agg(
        latitude=("latitude", "mean"),
        longitude=("longitude", "mean")
    )
)

# round to 2 decimals
county_geo["latitude"] = county_geo["latitude"].round(2)
county_geo["longitude"] = county_geo["longitude"].round(2)

county_load = county_load[["datetime_ending_ept", "county_name", "state_short_name", "distributed_load_mw"]].merge(county_geo, on="county_name", how="left").copy()

In [103]:
county_load = county_load[(county_load["datetime_ending_ept"] >= "2026-05-01")].copy()

In [104]:
county_load["datetime_ending_ept"] = pd.to_datetime(county_load["datetime_ending_ept"])

county_load_shadow_price = county_load.merge(
    aquahar_shadow_price[["datetime_ending_ept", "shadow_price"]],
    on=["datetime_ending_ept"],
    how="left"
)

county_load_shadow_price["shadow_price"] = county_load_shadow_price["shadow_price"].fillna(0)

In [105]:
# row-level filter
df_filtered = county_load_shadow_price[
    (county_load_shadow_price["shadow_price"].notna()) &
    (county_load_shadow_price["shadow_price"] != 0)
].copy()

# keep only counties with mean load > 20
county_load_shadow = df_filtered[
    df_filtered.groupby("county_name")["distributed_load_mw"].transform("max") > 20
].copy()

In [106]:
df = county_load_shadow.copy()

feature_cols = ["distributed_load_mw"] 

# 3. correlation between each feature and shadow_price for each nodename
corr_by_node = (
    df.groupby("county_name")
      .apply(lambda g: g[feature_cols].corrwith(g["shadow_price"]))
      .reset_index()
)

corr_by_node.sort_values(by="distributed_load_mw", ascending=False)

,county_name,distributed_load_mw
38,VA_Franklin,0.683092
24,VA_Campbell,0.669183
21,VA_Botetourt,0.668462
1,MD_Charles,0.653598
77,VA_Southampton,0.648935
...,...,...
52,VA_King William,0.080034
42,VA_Greensville,0.033585
74,VA_Rockbridge,0.014032
23,VA_Buckingham,-0.055313


In [107]:
corr_by_node[corr_by_node["county_name"] == "VA_Stafford"]

,county_name,distributed_load_mw
79,VA_Stafford,0.542801


### plot

In [108]:
county_load_shadow_price = county_load_shadow_price.merge(
    aquahar_shadow_price_DA[["datetime_ending_ept", "shadow_price_DA"]],
    on=["datetime_ending_ept"],
    how="left"
)

county_load_shadow_price["shadow_price_DA"] = county_load_shadow_price["shadow_price_DA"].fillna(0)

In [109]:
df = county_load_shadow_price.copy()

node = "VA_Stafford"

sub = df[df["county_name"] == node].copy()

# ensure datetime is sorted
sub["datetime_ending_ept"] = pd.to_datetime(sub["datetime_ending_ept"])
sub = sub.sort_values("datetime_ending_ept")

fig = go.Figure()

# Left y-axis: distributed load
fig.add_trace(
    go.Scatter(
        x=sub["datetime_ending_ept"],
        y=sub["distributed_load_mw"],
        name="Distributed Load (MW)",
        line=dict(color="blue"),
        yaxis="y1"
    )
)

# Right y-axis: shadow price
fig.add_trace(
    go.Scatter(
        x=sub["datetime_ending_ept"],
        y=sub["shadow_price"],
        name="Shadow Price",
        line=dict(color="red"),
        yaxis="y2"
    )
)

fig.add_trace(
    go.Scatter(
        x=sub["datetime_ending_ept"],
        y=sub["shadow_price_DA"],
        name="Shadow Price DA",
        line=dict(color="green"),
        yaxis="y2"
    )
)


# Layout with dual axis
fig.update_layout(
    title=f"{node}: Load vs Shadow Price",
    xaxis=dict(title="Datetime"),
    
    yaxis=dict(
        title="Distributed Load (MW)",
        side="left"
    ),
    
    yaxis2=dict(
        title="Shadow Price",
        overlaying="y",
        side="right"
    ),
    
    legend=dict(x=0.01, y=0.99),
    height=500,
    width=2000
)

fig.show()

# ELMONT

## shadow

In [83]:
# copy
df = RT_constr[RT_constr["MONITORED_FACILITY_DESC"].str.contains("ELMONT")].copy()

# hourly columns
hour_cols = [str(i) for i in range(1, 25)]

# keep only rows that have at least one hourly value
df = df.dropna(subset=hour_cols, how="all")

df["file_date"] = pd.to_datetime(df["file_date"])

# reshape to long format
elmont_shadow_price = df.melt(
    id_vars=["MONITORED_FACILITY_DESC", "file_date"],
    value_vars=hour_cols,
    var_name="hour_ending",
    value_name="shadow_price"
)

# drop null hourly values
elmont_shadow_price = elmont_shadow_price.dropna(subset=["shadow_price"]).copy()

# convert HE to int
elmont_shadow_price["hour_ending"] = elmont_shadow_price["hour_ending"].astype(int)

# build datetime_ending_ept
elmont_shadow_price["datetime_ending_ept"] = (
    elmont_shadow_price["file_date"] + pd.to_timedelta(
        elmont_shadow_price["hour_ending"], unit="h"
    )
)

# rename to match your target format
elmont_shadow_price = elmont_shadow_price.rename(
    columns={"MONITORED_FACILITY_DESC": "nodename"}
)

# reorder columns
elmont_shadow_price = elmont_shadow_price[
    ["datetime_ending_ept", "nodename", "shadow_price"]
].sort_values(
    ["nodename", "datetime_ending_ept"]
).drop_duplicates(
    subset=["nodename", "datetime_ending_ept"],
    keep="last"
).reset_index(drop=True)

elmont_shadow_price.head()

,datetime_ending_ept,nodename,shadow_price
0,2026-05-26 21:00:00,ELMONT4 TX1 XFORMER H 500 KV,207.57
1,2026-04-04 20:00:00,ELMONT4 TX2 XFORMER H 500 KV,1333.33
2,2026-04-04 21:00:00,ELMONT4 TX2 XFORMER H 500 KV,834.02
3,2026-04-04 22:00:00,ELMONT4 TX2 XFORMER H 500 KV,0.04
4,2026-04-15 19:00:00,ELMONT4 TX2 XFORMER H 500 KV,333.33


In [84]:
elmont_shadow_price['nodename'].value_counts()

nodename
ELMONT4  TX2      XFORMER   H  500 KV    24
ELMONT4 L192 CB                230 KV    19
ELMONT4  TX1      XFORMER   H  500 KV     1
Name: count, dtype: int64

In [85]:
elmont_shadow_price['shadow_price'].max()

np.float64(2000.0)

In [86]:
# copy
df = DA_constr[DA_constr["MONITORED_FACILITY_DESC"].str.contains("ELMONT")].copy()

# hourly columns
hour_cols = [str(i) for i in range(1, 25)]

# keep only rows that have at least one hourly value
df = df.dropna(subset=hour_cols, how="all")

df["file_date"] = pd.to_datetime(df["file_date"])

# reshape to long format
elmont_shadow_price_DA = df.melt(
    id_vars=["MONITORED_FACILITY_DESC", "file_date"],
    value_vars=hour_cols,
    var_name="hour_ending",
    value_name="shadow_price"
)

# drop null hourly values
elmont_shadow_price_DA = elmont_shadow_price_DA.dropna(subset=["shadow_price"]).copy()

# convert HE to int
elmont_shadow_price_DA["hour_ending"] = elmont_shadow_price_DA["hour_ending"].astype(int)

# build datetime_ending_ept
elmont_shadow_price_DA["datetime_ending_ept"] = (
    elmont_shadow_price_DA["file_date"] + 
    pd.to_timedelta(elmont_shadow_price_DA["hour_ending"], unit="h")
)

# rename to match your target format
elmont_shadow_price_DA = elmont_shadow_price_DA.rename(
    columns={
        "MONITORED_FACILITY_DESC": "nodename",
        "shadow_price": "shadow_price_DA"
    }
)

# reorder columns
elmont_shadow_price_DA = elmont_shadow_price_DA[
    ["datetime_ending_ept", "nodename", "shadow_price_DA"]
].sort_values(
    ["nodename", "datetime_ending_ept"]
).drop_duplicates(
    subset=["nodename", "datetime_ending_ept"],
    keep="last"
).reset_index(drop=True)

elmont_shadow_price_DA.head()

,datetime_ending_ept,nodename,shadow_price_DA
0,2026-05-04 16:00:00,ELMONT4_230KV2103_1_LN,13.21
1,2026-05-04 18:00:00,ELMONT4_230KV2103_1_LN,94.67
2,2026-05-05 11:00:00,ELMONT4_230KV2103_1_LN,39.12
3,2026-05-05 12:00:00,ELMONT4_230KV2103_1_LN,46.68
4,2026-05-05 14:00:00,ELMONT4_230KV2103_1_LN,28.54


## county

In [87]:
# Get all nodenames that belong to agg_nodename == "DOM" in hourly_load
dom_nodes = (
    hourly_load.loc[hourly_load["agg_nodename"] == "DOM", "nodename"]
    .dropna()
    .unique()
)

# Map those DOM nodenames to county_name using node_cty_map
dom_node_counties = (
    node_cty_map.loc[node_cty_map["nodename"].isin(dom_nodes), ["nodename", "county_name"]]
    .dropna(subset=["county_name"])
    .drop_duplicates()
)

# Get all county_name touched by those DOM nodenames
dom_counties = dom_node_counties["county_name"].unique()

# From those counties, get all nodenames in node_cty_map
all_nodes_in_dom_counties = (
    node_cty_map.loc[node_cty_map["county_name"].isin(dom_counties), "nodename"]
    .dropna()
    .unique()
)

# Select all rows from hourly_load whose nodename is in those nodenames
hourly_load_dom_counties = hourly_load.loc[
    hourly_load["nodename"].isin(all_nodes_in_dom_counties)
].copy()

In [88]:
hourly_load_county = hourly_load_dom_counties[["datetime_ending_ept", "agg_nodename", "nodename", "distribution_factor", "forecast_area", "forecast_load_mw", "distributed_load_mw"]].merge(node_cty_map[["nodename", "county_name", "latitude", "longitude", "state_short_name"]], on="nodename", how="left")

county_load = (
    hourly_load_county
    .groupby(
        ["datetime_ending_ept", "county_name", "state_short_name"],
        as_index=False
    )
    .agg(
        distributed_load_mw=("distributed_load_mw", "sum"),
        latitude=("latitude", "mean"),
        longitude=("longitude", "mean")
    )
    .sort_values(["datetime_ending_ept", "county_name"])
)

In [89]:
county_geo = (
    county_load
    .groupby("county_name", as_index=False)
    .agg(
        latitude=("latitude", "mean"),
        longitude=("longitude", "mean")
    )
)

# round to 2 decimals
county_geo["latitude"] = county_geo["latitude"].round(2)
county_geo["longitude"] = county_geo["longitude"].round(2)

county_load = county_load[["datetime_ending_ept", "county_name", "state_short_name", "distributed_load_mw"]].merge(county_geo, on="county_name", how="left").copy()

In [90]:
county_load = county_load[(county_load["datetime_ending_ept"] >= "2026-04-01")].copy()

In [91]:
county_load["datetime_ending_ept"] = pd.to_datetime(county_load["datetime_ending_ept"])

county_load_shadow_price = county_load.merge(
    elmont_shadow_price[["datetime_ending_ept", "shadow_price"]],
    on=["datetime_ending_ept"],
    how="left"
)

county_load_shadow_price["shadow_price"] = county_load_shadow_price["shadow_price"].fillna(0)

In [92]:
# row-level filter
df_filtered = county_load_shadow_price[
    (county_load_shadow_price["shadow_price"].notna()) &
    (county_load_shadow_price["shadow_price"] != 0)
].copy()

# keep only counties with mean load > 20
county_load_shadow = df_filtered[
    df_filtered.groupby("county_name")["distributed_load_mw"].transform("max") > 20
].copy()

In [93]:
df = county_load_shadow.copy()

feature_cols = ["distributed_load_mw"] 

# 3. correlation between each feature and shadow_price for each nodename
corr_by_node = (
    df.groupby("county_name")
      .apply(lambda g: g[feature_cols].corrwith(g["shadow_price"]))
      .reset_index()
)

corr_by_node.sort_values(by="distributed_load_mw", ascending=False)

,county_name,distributed_load_mw
30,VA_Chesapeake,0.452851
13,NC_Pasquotank,0.431014
89,WV_Jefferson,0.423517
82,VA_Suffolk,0.405277
21,VA_Bedford,0.396982
...,...,...
35,VA_Essex,-0.038287
11,NC_Nash,-0.071942
24,VA_Buckingham,-0.072652
34,VA_Dinwiddie,-0.114550


In [94]:
corr_by_node[corr_by_node["county_name"] == "VA_Hanover"]

,county_name,distributed_load_mw
47,VA_Hanover,0.264146


### plot

In [95]:
county_load_shadow_price = county_load_shadow_price.merge(
    elmont_shadow_price_DA[["datetime_ending_ept", "shadow_price_DA"]],
    on=["datetime_ending_ept"],
    how="left"
)

county_load_shadow_price["shadow_price_DA"] = county_load_shadow_price["shadow_price_DA"].fillna(0)

In [96]:
df = county_load_shadow_price.copy()

node = "VA_Hanover"

sub = df[df["county_name"] == node].copy()

# ensure datetime is sorted
sub["datetime_ending_ept"] = pd.to_datetime(sub["datetime_ending_ept"])
sub = sub.sort_values("datetime_ending_ept")

fig = go.Figure()

# Left y-axis: distributed load
fig.add_trace(
    go.Scatter(
        x=sub["datetime_ending_ept"],
        y=sub["distributed_load_mw"],
        name="Distributed Load (MW)",
        line=dict(color="blue"),
        yaxis="y1"
    )
)

# Right y-axis: shadow price
fig.add_trace(
    go.Scatter(
        x=sub["datetime_ending_ept"],
        y=sub["shadow_price"],
        name="Shadow Price",
        line=dict(color="red"),
        yaxis="y2"
    )
)

fig.add_trace(
    go.Scatter(
        x=sub["datetime_ending_ept"],
        y=sub["shadow_price_DA"],
        name="Shadow Price DA",
        line=dict(color="green"),
        yaxis="y2"
    )
)


# Layout with dual axis
fig.update_layout(
    title=f"{node}: Load vs Shadow Price",
    xaxis=dict(title="Datetime"),
    
    yaxis=dict(
        title="Distributed Load (MW)",
        side="left"
    ),
    
    yaxis2=dict(
        title="Shadow Price",
        overlaying="y",
        side="right"
    ),
    
    legend=dict(x=0.01, y=0.99),
    height=500,
    width=2000
)

fig.show()

# Haviland

## shadow

In [44]:
# copy
df = RT_constr[RT_constr["MONITORED_FACILITY_DESC"].str.contains("HAVILAND-PAULDING")].copy()

# hourly columns
hour_cols = [str(i) for i in range(1, 25)]

# keep only rows that have at least one hourly value
df = df.dropna(subset=hour_cols, how="all")

df["file_date"] = pd.to_datetime(df["file_date"])

# reshape to long format
haviland_shadow_price = df.melt(
    id_vars=["MONITORED_FACILITY_DESC", "file_date"],
    value_vars=hour_cols,
    var_name="hour_ending",
    value_name="shadow_price"
)

# drop null hourly values
haviland_shadow_price = haviland_shadow_price.dropna(subset=["shadow_price"]).copy()

# convert HE to int
haviland_shadow_price["hour_ending"] = haviland_shadow_price["hour_ending"].astype(int)

# build datetime_ending_ept
haviland_shadow_price["datetime_ending_ept"] = (
    haviland_shadow_price["file_date"] + pd.to_timedelta(
        haviland_shadow_price["hour_ending"], unit="h"
    )
)

# rename to match your target format
haviland_shadow_price = haviland_shadow_price.rename(
    columns={"MONITORED_FACILITY_DESC": "nodename"}
)

# reorder columns
haviland_shadow_price = haviland_shadow_price[
    ["datetime_ending_ept", "nodename", "shadow_price"]
].sort_values(
    ["nodename", "datetime_ending_ept"]
).drop_duplicates(
    subset=["nodename", "datetime_ending_ept"],
    keep="last"
).reset_index(drop=True)

haviland_shadow_price.head()

,datetime_ending_ept,nodename,shadow_price
0,2026-03-31 03:00:00,HAVILAND-PAULDING HAV-PAU A 69 KV,32.16
1,2026-03-31 05:00:00,HAVILAND-PAULDING HAV-PAU A 69 KV,43.52
2,2026-03-31 06:00:00,HAVILAND-PAULDING HAV-PAU A 69 KV,172.33
3,2026-03-31 07:00:00,HAVILAND-PAULDING HAV-PAU A 69 KV,450.54
4,2026-03-31 08:00:00,HAVILAND-PAULDING HAV-PAU A 69 KV,554.30


In [45]:
haviland_shadow_price['shadow_price'].max()

np.float64(979.48)

In [46]:
# copy
df = DA_constr[DA_constr["MONITORED_FACILITY_DESC"].str.contains("HAVILAND_69KVHAV-PAU")].copy()

# hourly columns
hour_cols = [str(i) for i in range(1, 25)]

# keep only rows that have at least one hourly value
df = df.dropna(subset=hour_cols, how="all")

df["file_date"] = pd.to_datetime(df["file_date"])

# reshape to long format
haviland_shadow_price_DA = df.melt(
    id_vars=["MONITORED_FACILITY_DESC", "file_date"],
    value_vars=hour_cols,
    var_name="hour_ending",
    value_name="shadow_price"
)

# drop null hourly values
haviland_shadow_price_DA = haviland_shadow_price_DA.dropna(subset=["shadow_price"]).copy()

# convert HE to int
haviland_shadow_price_DA["hour_ending"] = haviland_shadow_price_DA["hour_ending"].astype(int)

# build datetime_ending_ept
haviland_shadow_price_DA["datetime_ending_ept"] = (
    haviland_shadow_price_DA["file_date"] + 
    pd.to_timedelta(haviland_shadow_price_DA["hour_ending"], unit="h")
)

# rename to match your target format
haviland_shadow_price_DA = haviland_shadow_price_DA.rename(
    columns={
        "MONITORED_FACILITY_DESC": "nodename",
        "shadow_price": "shadow_price_DA"
    }
)

# reorder columns
haviland_shadow_price_DA = haviland_shadow_price_DA[
    ["datetime_ending_ept", "nodename", "shadow_price_DA"]
].sort_values(
    ["nodename", "datetime_ending_ept"]
).drop_duplicates(
    subset=["nodename", "datetime_ending_ept"],
    keep="last"
).reset_index(drop=True)

haviland_shadow_price_DA.head()

,datetime_ending_ept,nodename,shadow_price_DA
0,2026-04-01 10:00:00,HAVILAND_69KVHAV-PAU_1_LN,60.77
1,2026-04-01 11:00:00,HAVILAND_69KVHAV-PAU_1_LN,68.29
2,2026-04-01 12:00:00,HAVILAND_69KVHAV-PAU_1_LN,76.94
3,2026-04-01 13:00:00,HAVILAND_69KVHAV-PAU_1_LN,19.25
4,2026-04-01 14:00:00,HAVILAND_69KVHAV-PAU_1_LN,2.69


## county

In [47]:
# Get all nodenames that belong to agg_nodename == "AEP" in hourly_load
aep_nodes = (
    hourly_load.loc[hourly_load["agg_nodename"].isin(["AEP", "ATSI"]), "nodename"]
    .dropna()
    .unique()
)

# Map those AEP nodenames to county_name using node_cty_map
aep_node_counties = (
    node_cty_map.loc[node_cty_map["nodename"].isin(aep_nodes), ["nodename", "county_name"]]
    .dropna(subset=["county_name"])
    .drop_duplicates()
)

# Get all county_name touched by those AEP nodenames
aep_counties = aep_node_counties["county_name"].unique()

# From those counties, get all nodenames in node_cty_map
all_nodes_in_aep_counties = (
    node_cty_map.loc[node_cty_map["county_name"].isin(aep_counties), "nodename"]
    .dropna()
    .unique()
)

# Select all rows from hourly_load whose nodename is in those nodenames
hourly_load_aep_counties = hourly_load.loc[
    hourly_load["nodename"].isin(all_nodes_in_aep_counties)
].copy()

In [48]:
hourly_load_county = hourly_load_aep_counties[["datetime_ending_ept", "agg_nodename", "nodename", "distribution_factor", "forecast_area", "forecast_load_mw", "distributed_load_mw"]].merge(node_cty_map[["nodename", "county_name", "latitude", "longitude", "state_short_name"]], on="nodename", how="left")

county_load = (
    hourly_load_county
    .groupby(
        ["datetime_ending_ept", "county_name", "state_short_name"],
        as_index=False
    )
    .agg(
        distributed_load_mw=("distributed_load_mw", "sum"),
        latitude=("latitude", "mean"),
        longitude=("longitude", "mean")
    )
    .sort_values(["datetime_ending_ept", "county_name"])
)

In [49]:
county_geo = (
    county_load
    .groupby("county_name", as_index=False)
    .agg(
        latitude=("latitude", "mean"),
        longitude=("longitude", "mean")
    )
)

# round to 2 decimals
county_geo["latitude"] = county_geo["latitude"].round(2)
county_geo["longitude"] = county_geo["longitude"].round(2)

county_load = county_load[["datetime_ending_ept", "county_name", "state_short_name", "distributed_load_mw"]].merge(county_geo, on="county_name", how="left").copy()

In [50]:
county_load = county_load[county_load["state_short_name"] == "OH"].copy()

county_load = county_load[(county_load["datetime_ending_ept"] >= "2026-03-31")].copy()

In [51]:
county_load["datetime_ending_ept"] = pd.to_datetime(county_load["datetime_ending_ept"])

county_load_shadow_price = county_load.merge(
    haviland_shadow_price[["datetime_ending_ept", "shadow_price"]],
    on=["datetime_ending_ept"],
    how="left"
)

county_load_shadow_price["shadow_price"] = county_load_shadow_price["shadow_price"].fillna(0)

In [52]:
# row-level filter
df_filtered = county_load_shadow_price[
    (county_load_shadow_price["shadow_price"].notna()) &
    (county_load_shadow_price["shadow_price"] != 0)
].copy()

# keep only counties with mean load > 20
county_load_shadow = df_filtered[
    df_filtered.groupby("county_name")["distributed_load_mw"].transform("mean") > 20
].copy()

In [53]:
df = county_load_shadow.copy()

feature_cols = ["distributed_load_mw"] 

# 3. correlation between each feature and shadow_price for each nodename
corr_by_node = (
    df.groupby("county_name")
      .apply(lambda g: g[feature_cols].corrwith(g["shadow_price"]))
      .reset_index()
)

corr_by_node.sort_values(by="distributed_load_mw", ascending=False)

,county_name,distributed_load_mw
57,OH_Scioto,0.522555
38,OH_Mahoning,0.501190
63,OH_Tuscarawas,0.490130
18,OH_Fulton,0.486491
69,OH_Wood,0.475357
...,...,...
53,OH_Putnam,0.082505
6,OH_Brown,0.033727
70,OH_Wyandot,0.012331
5,OH_Belmont,-0.001700


In [54]:
corr_by_node[corr_by_node["county_name"] == "OH_Paulding"]

,county_name,distributed_load_mw
48,OH_Paulding,0.170574


In [55]:
corr_by_node[corr_by_node["county_name"] == "OH_Van Wert"]

,county_name,distributed_load_mw
65,OH_Van Wert,0.370711


### plot

In [56]:
county_load_shadow_price = county_load_shadow_price.merge(
    haviland_shadow_price_DA[["datetime_ending_ept", "shadow_price_DA"]],
    on=["datetime_ending_ept"],
    how="left"
)

county_load_shadow_price["shadow_price_DA"] = county_load_shadow_price["shadow_price_DA"].fillna(0)



In [57]:
df = county_load_shadow_price.copy()

node = "OH_Van Wert"

sub = df[df["county_name"] == node].copy()

# ensure datetime is sorted
sub["datetime_ending_ept"] = pd.to_datetime(sub["datetime_ending_ept"])
sub = sub.sort_values("datetime_ending_ept")

fig = go.Figure()

# Left y-axis: distributed load
fig.add_trace(
    go.Scatter(
        x=sub["datetime_ending_ept"],
        y=sub["distributed_load_mw"],
        name="Distributed Load (MW)",
        line=dict(color="blue"),
        yaxis="y1"
    )
)

# Right y-axis: shadow price
fig.add_trace(
    go.Scatter(
        x=sub["datetime_ending_ept"],
        y=sub["shadow_price"],
        name="Shadow Price",
        line=dict(color="red"),
        yaxis="y2"
    )
)


fig.add_trace(
    go.Scatter(
        x=sub["datetime_ending_ept"],
        y=sub["shadow_price_DA"],
        name="Shadow Price DA",
        line=dict(color="green"),
        yaxis="y2"
    )
)

# Layout with dual axis
fig.update_layout(
    title=f"{node}: Load vs Shadow Price",
    xaxis=dict(title="Datetime"),
    
    yaxis=dict(
        title="Distributed Load (MW)",
        side="left"
    ),
    
    yaxis2=dict(
        title="Shadow Price",
        overlaying="y",
        side="right"
    ),
    
    legend=dict(x=0.01, y=0.99),
    height=500,
    width=2000
)

fig.show()

# Greenway

## shadow

In [72]:
# copy
df = RT_constr[RT_constr["MONITORED_FACILITY_DESC"].str.contains("GREENWAY")].copy()

# hourly columns
hour_cols = [str(i) for i in range(1, 25)]

# keep only rows that have at least one hourly value
df = df.dropna(subset=hour_cols, how="all")

df["file_date"] = pd.to_datetime(df["file_date"])

# reshape to long format
greenway_shadow_price = df.melt(
    id_vars=["MONITORED_FACILITY_DESC", "file_date"],
    value_vars=hour_cols,
    var_name="hour_ending",
    value_name="shadow_price"
)

# drop null hourly values
greenway_shadow_price = greenway_shadow_price.dropna(subset=["shadow_price"]).copy()

# convert HE to int
greenway_shadow_price["hour_ending"] = greenway_shadow_price["hour_ending"].astype(int)

# build datetime_ending_ept
greenway_shadow_price["datetime_ending_ept"] = (
    greenway_shadow_price["file_date"] + pd.to_timedelta(
        greenway_shadow_price["hour_ending"], unit="h"
    )
)

# rename to match your target format
greenway_shadow_price = greenway_shadow_price.rename(
    columns={"MONITORED_FACILITY_DESC": "nodename"}
)

# reorder columns
greenway_shadow_price = greenway_shadow_price[
    ["datetime_ending_ept", "nodename", "shadow_price"]
].sort_values(
    ["nodename", "datetime_ending_ept"]
).drop_duplicates(
    subset=["nodename", "datetime_ending_ept"],
    keep="last"
).reset_index(drop=True)

greenway_shadow_price.head()

,datetime_ending_ept,nodename,shadow_price
0,2026-05-18 15:00:00,GREENWAY-ROUNDTBL 2031A1 A 230 KV,500.00
1,2026-05-18 16:00:00,GREENWAY-ROUNDTBL 2031A1 A 230 KV,1166.67
2,2026-05-19 12:00:00,GREENWAY-ROUNDTBL 2031A1 A 230 KV,1700.90
3,2026-05-19 13:00:00,GREENWAY-ROUNDTBL 2031A1 A 230 KV,2000.00
4,2026-05-19 14:00:00,GREENWAY-ROUNDTBL 2031A1 A 230 KV,1833.33


In [73]:
# copy
df = DA_constr[DA_constr["MONITORED_FACILITY_DESC"].str.contains("GREENWAY")].copy()

# hourly columns
hour_cols = [str(i) for i in range(1, 25)]

# keep only rows that have at least one hourly value
df = df.dropna(subset=hour_cols, how="all")

df["file_date"] = pd.to_datetime(df["file_date"])

# reshape to long format
greenway_shadow_price_DA = df.melt(
    id_vars=["MONITORED_FACILITY_DESC", "file_date"],
    value_vars=hour_cols,
    var_name="hour_ending",
    value_name="shadow_price"
)

# drop null hourly values
greenway_shadow_price_DA = greenway_shadow_price_DA.dropna(subset=["shadow_price"]).copy()

# convert HE to int
greenway_shadow_price_DA["hour_ending"] = greenway_shadow_price_DA["hour_ending"].astype(int)

# build datetime_ending_ept
greenway_shadow_price_DA["datetime_ending_ept"] = (
    greenway_shadow_price_DA["file_date"] + 
    pd.to_timedelta(greenway_shadow_price_DA["hour_ending"], unit="h")
)

# rename to match your target format
greenway_shadow_price_DA = greenway_shadow_price_DA.rename(
    columns={
        "MONITORED_FACILITY_DESC": "nodename",
        "shadow_price": "shadow_price_DA"
    }
)

# reorder columns
greenway_shadow_price_DA = greenway_shadow_price_DA[
    ["datetime_ending_ept", "nodename", "shadow_price_DA"]
].sort_values(
    ["nodename", "datetime_ending_ept"]
).drop_duplicates(
    subset=["nodename", "datetime_ending_ept"],
    keep="last"
).reset_index(drop=True)

greenway_shadow_price_DA.head()

,datetime_ending_ept,nodename,shadow_price_DA


In [74]:
greenway_shadow_price["shadow_price"].max()

np.float64(2000.0)

## county

In [75]:
# Get all nodenames that belong to agg_nodename == "DOM" in hourly_load
dom_nodes = (
    hourly_load.loc[hourly_load["agg_nodename"] == "DOM", "nodename"]
    .dropna()
    .unique()
)

# Map those DOM nodenames to county_name using node_cty_map
dom_node_counties = (
    node_cty_map.loc[node_cty_map["nodename"].isin(dom_nodes), ["nodename", "county_name"]]
    .dropna(subset=["county_name"])
    .drop_duplicates()
)

# Get all county_name touched by those DOM nodenames
dom_counties = dom_node_counties["county_name"].unique()

# From those counties, get all nodenames in node_cty_map
all_nodes_in_dom_counties = (
    node_cty_map.loc[node_cty_map["county_name"].isin(dom_counties), "nodename"]
    .dropna()
    .unique()
)

# Select all rows from hourly_load whose nodename is in those nodenames
hourly_load_dom_counties = hourly_load.loc[
    hourly_load["nodename"].isin(all_nodes_in_dom_counties)
].copy()

In [76]:
hourly_load_county = hourly_load_dom_counties[["datetime_ending_ept", "agg_nodename", "nodename", "distribution_factor", "forecast_area", "forecast_load_mw", "distributed_load_mw"]].merge(node_cty_map[["nodename", "county_name", "latitude", "longitude", "state_short_name"]], on="nodename", how="left")

county_load = (
    hourly_load_county
    .groupby(
        ["datetime_ending_ept", "county_name", "state_short_name"],
        as_index=False
    )
    .agg(
        distributed_load_mw=("distributed_load_mw", "sum"),
        latitude=("latitude", "mean"),
        longitude=("longitude", "mean")
    )
    .sort_values(["datetime_ending_ept", "county_name"])
)

In [77]:
county_geo = (
    county_load
    .groupby("county_name", as_index=False)
    .agg(
        latitude=("latitude", "mean"),
        longitude=("longitude", "mean")
    )
)

# round to 2 decimals
county_geo["latitude"] = county_geo["latitude"].round(2)
county_geo["longitude"] = county_geo["longitude"].round(2)

county_load = county_load[["datetime_ending_ept", "county_name", "state_short_name", "distributed_load_mw"]].merge(county_geo, on="county_name", how="left").copy()

In [78]:
county_load = county_load[(county_load["datetime_ending_ept"] >= "2026-05-01")].copy()

In [79]:
county_load["datetime_ending_ept"] = pd.to_datetime(county_load["datetime_ending_ept"])

county_load_shadow_price = county_load.merge(
    greenway_shadow_price[["datetime_ending_ept", "shadow_price"]],
    on=["datetime_ending_ept"],
    how="left"
)

county_load_shadow_price["shadow_price"] = county_load_shadow_price["shadow_price"].fillna(0)

In [80]:
# row-level filter
df_filtered = county_load_shadow_price[
    (county_load_shadow_price["shadow_price"].notna()) &
    (county_load_shadow_price["shadow_price"] != 0)
].copy()

# keep only counties with mean load > 20
county_load_shadow = df_filtered[
    df_filtered.groupby("county_name")["distributed_load_mw"].transform("max") > 20
].copy()

### plot

In [81]:
county_load_shadow_price = county_load_shadow_price.merge(
    greenway_shadow_price_DA[["datetime_ending_ept", "shadow_price_DA"]],
    on=["datetime_ending_ept"],
    how="left"
)

county_load_shadow_price["shadow_price_DA"] = county_load_shadow_price["shadow_price_DA"].fillna(0)

In [82]:
df = county_load_shadow_price.copy()

node = "VA_Loudoun"

sub = df[df["county_name"] == node].copy()

# ensure datetime is sorted
sub["datetime_ending_ept"] = pd.to_datetime(sub["datetime_ending_ept"])
sub = sub.sort_values("datetime_ending_ept")

fig = go.Figure()

# Left y-axis: distributed load
fig.add_trace(
    go.Scatter(
        x=sub["datetime_ending_ept"],
        y=sub["distributed_load_mw"],
        name="Distributed Load (MW)",
        line=dict(color="blue"),
        yaxis="y1"
    )
)

# Right y-axis: shadow price
fig.add_trace(
    go.Scatter(
        x=sub["datetime_ending_ept"],
        y=sub["shadow_price"],
        name="Shadow Price",
        line=dict(color="red"),
        yaxis="y2"
    )
)

fig.add_trace(
    go.Scatter(
        x=sub["datetime_ending_ept"],
        y=sub["shadow_price_DA"],
        name="Shadow Price DA",
        line=dict(color="green"),
        yaxis="y2"
    )
)

# Layout with dual axis
fig.update_layout(
    title=f"{node}: Load vs Shadow Price",
    xaxis=dict(title="Datetime"),
    
    yaxis=dict(
        title="Distributed Load (MW)",
        side="left"
    ),
    
    yaxis2=dict(
        title="Shadow Price",
        overlaying="y",
        side="right"
    ),
    
    legend=dict(x=0.01, y=0.99),
    height=500,
    width=2000
)

fig.show()

# ASHBURN

## shadow

In [203]:
# copy
df = RT_constr[RT_constr["MONITORED_FACILITY_DESC"].str.contains("ASHBURN-GOOSECRE")].copy()

# hourly columns
hour_cols = [str(i) for i in range(1, 25)]

# keep only rows that have at least one hourly value
df = df.dropna(subset=hour_cols, how="all")

df["file_date"] = pd.to_datetime(df["file_date"])

# reshape to long format
ashburn_shadow_price = df.melt(
    id_vars=["MONITORED_FACILITY_DESC", "file_date"],
    value_vars=hour_cols,
    var_name="hour_ending",
    value_name="shadow_price"
)

# drop null hourly values
ashburn_shadow_price = ashburn_shadow_price.dropna(subset=["shadow_price"]).copy()

# convert HE to int
ashburn_shadow_price["hour_ending"] = ashburn_shadow_price["hour_ending"].astype(int)

# build datetime_ending_ept
ashburn_shadow_price["datetime_ending_ept"] = (
    ashburn_shadow_price["file_date"] + pd.to_timedelta(
        ashburn_shadow_price["hour_ending"], unit="h"
    )
)

# rename to match your target format
ashburn_shadow_price = ashburn_shadow_price.rename(
    columns={"MONITORED_FACILITY_DESC": "nodename"}
)

# reorder columns
ashburn_shadow_price = ashburn_shadow_price[
    ["datetime_ending_ept", "nodename", "shadow_price"]
].sort_values(
    ["nodename", "datetime_ending_ept"]
).drop_duplicates(
    subset=["nodename", "datetime_ending_ept"],
    keep="last"
).reset_index(drop=True)

ashburn_shadow_price.head()

,datetime_ending_ept,nodename,shadow_price
0,2026-05-26 01:00:00,ASHBURN-GOOSECRE 227D B 230 KV,17.05
1,2026-05-26 07:00:00,ASHBURN-GOOSECRE 227D B 230 KV,165.52
2,2026-05-26 08:00:00,ASHBURN-GOOSECRE 227D B 230 KV,105.07
3,2026-05-26 09:00:00,ASHBURN-GOOSECRE 227D B 230 KV,1217.65
4,2026-05-26 10:00:00,ASHBURN-GOOSECRE 227D B 230 KV,2000.00


In [207]:
# copy
df = DA_constr[DA_constr["MONITORED_FACILITY_DESC"].str.contains("ASHBURN-GOOSECRE")].copy()

# hourly columns
hour_cols = [str(i) for i in range(1, 25)]

# keep only rows that have at least one hourly value
df = df.dropna(subset=hour_cols, how="all")

df["file_date"] = pd.to_datetime(df["file_date"])

# reshape to long format
ashburn_shadow_price_DA = df.melt(
    id_vars=["MONITORED_FACILITY_DESC", "file_date"],
    value_vars=hour_cols,
    var_name="hour_ending",
    value_name="shadow_price"
)

# drop null hourly values
ashburn_shadow_price_DA = ashburn_shadow_price_DA.dropna(subset=["shadow_price"]).copy()

# convert HE to int
ashburn_shadow_price_DA["hour_ending"] = ashburn_shadow_price_DA["hour_ending"].astype(int)

# build datetime_ending_ept
ashburn_shadow_price_DA["datetime_ending_ept"] = (
    ashburn_shadow_price_DA["file_date"] + 
    pd.to_timedelta(ashburn_shadow_price_DA["hour_ending"], unit="h")
)

# rename to match your target format
ashburn_shadow_price_DA = ashburn_shadow_price_DA.rename(
    columns={
        "MONITORED_FACILITY_DESC": "nodename",
        "shadow_price": "shadow_price_DA"
    }
)

# reorder columns
ashburn_shadow_price_DA = ashburn_shadow_price_DA[
    ["datetime_ending_ept", "nodename", "shadow_price_DA"]
].sort_values(
    ["nodename", "datetime_ending_ept"]
).drop_duplicates(
    subset=["nodename", "datetime_ending_ept"],
    keep="last"
).reset_index(drop=True)

ashburn_shadow_price_DA.head()

,datetime_ending_ept,nodename,shadow_price_DA
0,2026-04-02 08:00:00,ASHBURN-GOOSECRE 227D B 230 KV,15.79
1,2026-05-19 14:00:00,ASHBURN-GOOSECRE 227D B 230 KV,656.83
2,2026-05-19 16:00:00,ASHBURN-GOOSECRE 227D B 230 KV,1488.65
3,2026-05-19 17:00:00,ASHBURN-GOOSECRE 227D B 230 KV,2000.00
4,2026-05-19 18:00:00,ASHBURN-GOOSECRE 227D B 230 KV,2000.00


In [208]:
ashburn_shadow_price["shadow_price"].max()

np.float64(2000.0)

## county

In [209]:
# Get all nodenames that belong to agg_nodename == "DOM" in hourly_load
dom_nodes = (
    hourly_load.loc[hourly_load["agg_nodename"] == "DOM", "nodename"]
    .dropna()
    .unique()
)

# Map those DOM nodenames to county_name using node_cty_map
dom_node_counties = (
    node_cty_map.loc[node_cty_map["nodename"].isin(dom_nodes), ["nodename", "county_name"]]
    .dropna(subset=["county_name"])
    .drop_duplicates()
)

# Get all county_name touched by those DOM nodenames
dom_counties = dom_node_counties["county_name"].unique()

# From those counties, get all nodenames in node_cty_map
all_nodes_in_dom_counties = (
    node_cty_map.loc[node_cty_map["county_name"].isin(dom_counties), "nodename"]
    .dropna()
    .unique()
)

# Select all rows from hourly_load whose nodename is in those nodenames
hourly_load_dom_counties = hourly_load.loc[
    hourly_load["nodename"].isin(all_nodes_in_dom_counties)
].copy()

In [210]:
hourly_load_county = hourly_load_dom_counties[["datetime_ending_ept", "agg_nodename", "nodename", "distribution_factor", "forecast_area", "forecast_load_mw", "distributed_load_mw"]].merge(node_cty_map[["nodename", "county_name", "latitude", "longitude", "state_short_name"]], on="nodename", how="left")

county_load = (
    hourly_load_county
    .groupby(
        ["datetime_ending_ept", "county_name", "state_short_name"],
        as_index=False
    )
    .agg(
        distributed_load_mw=("distributed_load_mw", "sum"),
        latitude=("latitude", "mean"),
        longitude=("longitude", "mean")
    )
    .sort_values(["datetime_ending_ept", "county_name"])
)

In [211]:
county_geo = (
    county_load
    .groupby("county_name", as_index=False)
    .agg(
        latitude=("latitude", "mean"),
        longitude=("longitude", "mean")
    )
)

# round to 2 decimals
county_geo["latitude"] = county_geo["latitude"].round(2)
county_geo["longitude"] = county_geo["longitude"].round(2)

county_load = county_load[["datetime_ending_ept", "county_name", "state_short_name", "distributed_load_mw"]].merge(county_geo, on="county_name", how="left").copy()

In [212]:
county_load = county_load[(county_load["datetime_ending_ept"] >= "2026-05-01")].copy()

In [213]:
county_load["datetime_ending_ept"] = pd.to_datetime(county_load["datetime_ending_ept"])

county_load_shadow_price = county_load.merge(
    ashburn_shadow_price[["datetime_ending_ept", "shadow_price"]],
    on=["datetime_ending_ept"],
    how="left"
)

county_load_shadow_price["shadow_price"] = county_load_shadow_price["shadow_price"].fillna(0)

In [214]:
county_load_shadow_price.head()

,datetime_ending_ept,county_name,state_short_name,distributed_load_mw,latitude,longitude,shadow_price
0,2026-05-01,DC_District of Columbia,DC,61.58026,38.89,-77.01,0.0
1,2026-05-01,MD_Montgomery,MD,1213.32219,39.07,-77.21,0.0
2,2026-05-01,NC_Beaufort,NC,14.06267,35.60,-76.88,0.0
3,2026-05-01,NC_Bertie,NC,24.72181,36.01,-76.96,0.0
4,2026-05-01,NC_Chowan,NC,0.46172,36.05,-76.48,0.0


In [215]:
# row-level filter
df_filtered = county_load_shadow_price[
    (county_load_shadow_price["shadow_price"].notna()) &
    (county_load_shadow_price["shadow_price"] != 0)
].copy()

# keep only counties with mean load > 20
county_load_shadow = df_filtered[
    df_filtered.groupby("county_name")["distributed_load_mw"].transform("max") > 20
].copy()

In [216]:
df = county_load_shadow.copy()

feature_cols = ["distributed_load_mw"] 

# 3. correlation between each feature and shadow_price for each nodename
corr_by_node = (
    df.groupby("county_name")
      .apply(lambda g: g[feature_cols].corrwith(g["shadow_price"]))
      .reset_index()
)

corr_by_node.sort_values(by="distributed_load_mw", ascending=False)

,county_name,distributed_load_mw
77,VA_Southampton,0.717918
65,VA_Pittsylvania,0.691528
84,VA_Waynesboro,0.680400
62,VA_Nottoway,0.663863
61,VA_Norfolk,0.652510
...,...,...
56,VA_Manassas Park,-0.265800
10,NC_Martin,-0.270032
55,VA_Manassas,-0.271605
4,NC_Bertie,-0.292707


### plot

In [217]:
county_load_shadow_price = county_load_shadow_price.merge(
    ashburn_shadow_price_DA[["datetime_ending_ept", "shadow_price_DA"]],
    on=["datetime_ending_ept"],
    how="left"
)

county_load_shadow_price["shadow_price_DA"] = county_load_shadow_price["shadow_price_DA"].fillna(0)

In [218]:
df = county_load_shadow_price.copy()

node = "VA_Fairfax"

sub = df[df["county_name"] == node].copy()

# ensure datetime is sorted
sub["datetime_ending_ept"] = pd.to_datetime(sub["datetime_ending_ept"])
sub = sub.sort_values("datetime_ending_ept")

fig = go.Figure()

# Left y-axis: distributed load
fig.add_trace(
    go.Scatter(
        x=sub["datetime_ending_ept"],
        y=sub["distributed_load_mw"],
        name="Distributed Load (MW)",
        line=dict(color="blue"),
        yaxis="y1"
    )
)

# Right y-axis: shadow price
fig.add_trace(
    go.Scatter(
        x=sub["datetime_ending_ept"],
        y=sub["shadow_price"],
        name="Shadow Price",
        line=dict(color="red"),
        yaxis="y2"
    )
)

fig.add_trace(
    go.Scatter(
        x=sub["datetime_ending_ept"],
        y=sub["shadow_price_DA"],
        name="Shadow Price DA",
        line=dict(color="green"),
        yaxis="y2"
    )
)

# Layout with dual axis
fig.update_layout(
    title=f"{node}: Load vs Shadow Price",
    xaxis=dict(title="Datetime"),
    
    yaxis=dict(
        title="Distributed Load (MW)",
        side="left"
    ),
    
    yaxis2=dict(
        title="Shadow Price",
        overlaying="y",
        side="right"
    ),
    
    legend=dict(x=0.01, y=0.99),
    height=500,
    width=2000
)

fig.show()

# NEWCARLI-OLIVE

In [74]:
# copy
df = RT_constr[
    RT_constr["MONITORED_FACILITY_DESC"].str.contains("NEWCARLI")
].copy()

# hourly columns
hour_cols = [str(i) for i in range(1, 25)]

# keep only rows that have at least one hourly value
df = df.dropna(subset=hour_cols, how="all")

df["file_date"] = pd.to_datetime(df["file_date"])

# reshape to long format
cook_shadow_price = df.melt(
    id_vars=["MONITORED_FACILITY_DESC", "file_date"],
    value_vars=hour_cols,
    var_name="hour_ending",
    value_name="shadow_price"
)

# drop null hourly values
cook_shadow_price = cook_shadow_price.dropna(
    subset=["shadow_price"]
).copy()

# convert HE to int
cook_shadow_price["hour_ending"] = (
    cook_shadow_price["hour_ending"].astype(int)
)

# build datetime_ending_ept
cook_shadow_price["datetime_ending_ept"] = (
    cook_shadow_price["file_date"]
    + pd.to_timedelta(
        cook_shadow_price["hour_ending"],
        unit="h"
    )
)

# rename to match your target format
cook_shadow_price = cook_shadow_price.rename(
    columns={"MONITORED_FACILITY_DESC": "nodename"}
)

# reorder columns
cook_shadow_price = cook_shadow_price[
    ["datetime_ending_ept", "nodename", "shadow_price"]
].sort_values(
    ["nodename", "datetime_ending_ept"]
).drop_duplicates(
    subset=["nodename", "datetime_ending_ept"],
    keep="last"
).reset_index(drop=True)

cook_shadow_price.head()

,datetime_ending_ept,nodename,shadow_price
0,2026-05-26 13:00:00,NEWCARLI-OLIVE B 138 KV,204.27
1,2026-05-26 14:00:00,NEWCARLI-OLIVE B 138 KV,479.31
2,2026-05-26 15:00:00,NEWCARLI-OLIVE B 138 KV,318.59
3,2026-05-26 16:00:00,NEWCARLI-OLIVE B 138 KV,586.91
4,2026-05-26 17:00:00,NEWCARLI-OLIVE B 138 KV,718.77


In [75]:
# copy
df = DA_constr[
    DA_constr["MONITORED_FACILITY_DESC"].str.contains("NEWCARLI")
].copy()

# hourly columns
hour_cols = [str(i) for i in range(1, 25)]

# keep only rows that have at least one hourly value
df = df.dropna(subset=hour_cols, how="all")

df["file_date"] = pd.to_datetime(df["file_date"])

# reshape to long format
cook_shadow_price_DA = df.melt(
    id_vars=["MONITORED_FACILITY_DESC", "file_date"],
    value_vars=hour_cols,
    var_name="hour_ending",
    value_name="shadow_price"
)

# drop null hourly values
cook_shadow_price_DA = cook_shadow_price_DA.dropna(
    subset=["shadow_price"]
).copy()

# convert HE to int
cook_shadow_price_DA["hour_ending"] = (
    cook_shadow_price_DA["hour_ending"].astype(int)
)

# build datetime_ending_ept
cook_shadow_price_DA["datetime_ending_ept"] = (
    cook_shadow_price_DA["file_date"]
    + pd.to_timedelta(
        cook_shadow_price_DA["hour_ending"],
        unit="h"
    )
)

# rename to match your target format
cook_shadow_price_DA = cook_shadow_price_DA.rename(
    columns={
        "MONITORED_FACILITY_DESC": "nodename",
        "shadow_price": "shadow_price_DA"
    }
)

# reorder columns
cook_shadow_price_DA = cook_shadow_price_DA[
    ["datetime_ending_ept", "nodename", "shadow_price_DA"]
].sort_values(
    ["nodename", "datetime_ending_ept"]
).drop_duplicates(
    subset=["nodename", "datetime_ending_ept"],
    keep="last"
).reset_index(drop=True)

cook_shadow_price_DA.head()

,datetime_ending_ept,nodename,shadow_price_DA
0,2026-05-28 15:00:00,NEWCARLI_138KVNEW-OLI1_1_LN,2.48
1,2026-05-28 16:00:00,NEWCARLI_138KVNEW-OLI1_1_LN,39.59


In [76]:
# Get all nodenames that belong to agg_nodename == "AEP" in hourly_load
aep_nodes = (
    hourly_load.loc[hourly_load["agg_nodename"].isin(["AEP"]), "nodename"]
    .dropna()
    .unique()
)

# Map those AEP nodenames to county_name using node_cty_map
aep_node_counties = (
    node_cty_map.loc[node_cty_map["nodename"].isin(aep_nodes), ["nodename", "county_name"]]
    .dropna(subset=["county_name"])
    .drop_duplicates()
)

# Get all county_name touched by those AEP nodenames
aep_counties = aep_node_counties["county_name"].unique()

# From those counties, get all nodenames in node_cty_map
all_nodes_in_aep_counties = (
    node_cty_map.loc[node_cty_map["county_name"].isin(aep_counties), "nodename"]
    .dropna()
    .unique()
)

# Select all rows from hourly_load whose nodename is in those nodenames
hourly_load_aep_counties = hourly_load.loc[
    hourly_load["nodename"].isin(all_nodes_in_aep_counties)
].copy()

In [77]:
hourly_load_county = hourly_load_aep_counties[["datetime_ending_ept", "agg_nodename", "nodename", "distribution_factor", "forecast_area", "forecast_load_mw", "distributed_load_mw"]].merge(node_cty_map[["nodename", "county_name", "latitude", "longitude", "state_short_name"]], on="nodename", how="left")

county_load = (
    hourly_load_county
    .groupby(
        ["datetime_ending_ept", "county_name", "state_short_name"],
        as_index=False
    )
    .agg(
        distributed_load_mw=("distributed_load_mw", "sum"),
        latitude=("latitude", "mean"),
        longitude=("longitude", "mean")
    )
    .sort_values(["datetime_ending_ept", "county_name"])
)

In [78]:
county_geo = (
    county_load
    .groupby("county_name", as_index=False)
    .agg(
        latitude=("latitude", "mean"),
        longitude=("longitude", "mean")
    )
)

# round to 2 decimals
county_geo["latitude"] = county_geo["latitude"].round(2)
county_geo["longitude"] = county_geo["longitude"].round(2)

county_load = county_load[["datetime_ending_ept", "county_name", "state_short_name", "distributed_load_mw"]].merge(county_geo, on="county_name", how="left").copy()

In [79]:
county_load = county_load[county_load["state_short_name"].isin(["IN", "MI"])].copy()

county_load = county_load[(county_load["datetime_ending_ept"] >= "2026-05-01")].copy()

In [80]:
county_load["datetime_ending_ept"] = pd.to_datetime(county_load["datetime_ending_ept"])

county_load_shadow_price = county_load.merge(
    cook_shadow_price[["datetime_ending_ept", "shadow_price"]],
    on=["datetime_ending_ept"],
    how="left"
)

county_load_shadow_price["shadow_price"] = county_load_shadow_price["shadow_price"].fillna(0)

In [81]:
# row-level filter
df_filtered = county_load_shadow_price[
    (county_load_shadow_price["shadow_price"].notna()) &
    (county_load_shadow_price["shadow_price"] != 0)
].copy()

# keep only counties with mean load > 20
county_load_shadow = df_filtered[
    df_filtered.groupby("county_name")["distributed_load_mw"].transform("mean") > 20
].copy()

In [82]:
df = county_load_shadow.copy()

feature_cols = ["distributed_load_mw"] 

# 3. correlation between each feature and shadow_price for each nodename
corr_by_node = (
    df.groupby("county_name")
      .apply(lambda g: g[feature_cols].corrwith(g["shadow_price"]))
      .reset_index()
)

corr_by_node.sort_values(by="distributed_load_mw", ascending=False)

,county_name,distributed_load_mw
9,IN_Kosciusko,0.297700
13,IN_Randolph,0.166011
15,IN_Wayne,0.074362
16,IN_Wells,0.014933
10,IN_LaPorte,0.011990
0,IN_Adams,0.003065
14,IN_St. Joseph,-0.009191
2,IN_Blackford,-0.031087
5,IN_Elkhart,-0.094739
19,MI_Cass,-0.096745


In [83]:
county_load_shadow_price = county_load_shadow_price.merge(
    cook_shadow_price_DA[["datetime_ending_ept", "shadow_price_DA"]],
    on=["datetime_ending_ept"],
    how="left"
)

county_load_shadow_price["shadow_price_DA"] = county_load_shadow_price["shadow_price_DA"].fillna(0)



In [84]:
df = county_load_shadow_price.copy()

node = "MI_Berrien"

sub = df[df["county_name"] == node].copy()

# ensure datetime is sorted
sub["datetime_ending_ept"] = pd.to_datetime(sub["datetime_ending_ept"])
sub = sub.sort_values("datetime_ending_ept")

fig = go.Figure()

# Left y-axis: distributed load
fig.add_trace(
    go.Scatter(
        x=sub["datetime_ending_ept"],
        y=sub["distributed_load_mw"],
        name="Distributed Load (MW)",
        line=dict(color="blue"),
        yaxis="y1"
    )
)

# Right y-axis: shadow price
fig.add_trace(
    go.Scatter(
        x=sub["datetime_ending_ept"],
        y=sub["shadow_price"],
        name="Shadow Price",
        line=dict(color="red"),
        yaxis="y2"
    )
)


fig.add_trace(
    go.Scatter(
        x=sub["datetime_ending_ept"],
        y=sub["shadow_price_DA"],
        name="Shadow Price DA",
        line=dict(color="green"),
        yaxis="y2"
    )
)

# Layout with dual axis
fig.update_layout(
    title=f"{node}: Load vs Shadow Price",
    xaxis=dict(title="Datetime"),
    
    yaxis=dict(
        title="Distributed Load (MW)",
        side="left"
    ),
    
    yaxis2=dict(
        title="Shadow Price",
        overlaying="y",
        side="right"
    ),
    
    legend=dict(x=0.01, y=0.99),
    height=500,
    width=2000
)

fig.show()